In [ ]:
#| default_exp game/food

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.core import Terrain, DrainageBasins
from HexMagic.styles import StyleCSS, SVGBuilder
from HexMagic.plot.primitives import  MapCord , PrimitiveDemo
from HexMagic.plot.hex import Hex, HexWrapper
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord ,HexDragMap, HexTouchMap, HexRegion
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import TerrainDisplay, CreamOverlay, RiverOverlay, ClimateOverlay
from HexMagic.water.soil import SoilSystem

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
import fasthtml.components as fc
import httpx
import random
import pandas as pd
import threading
from dataclasses import dataclass
from datetime import datetime
import numpy as np
import math
from dataclasses import dataclass, field
#| export
import heapq

In [ ]:
#| export

from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList
from HexMagic.game.piece import CountryFlag, DiagramGlyphs,PieceBoard, PieceStep, _move_cost,PieceBoardPlan
from scipy.optimize import linear_sum_assignment
from enum import Enum

In [ ]:
myTerr = TerraDemo().japan_korea_map()
rivers = myTerr.carve_to_ocean(num_lakes=0)

In [ ]:
myTerr.compute_climate()

In [ ]:
basins = DrainageBasins(myTerr)

In [ ]:
#TerrainDisplay(TerrainOverlay(),ClimateOverlay(),terrain=myTerr)

In [ ]:
!cat ../../HexMagic/water/*.py

In [ ]:
flag = CountryFlag.seaborn("husl", 1)[0]
flag.patternIndex = random.randint(0, 15)

canvas = SVGBuilder()
canvas.add_font('Cinzel')

# Three tiers with their typical scales
tiers = [
    ('board',  0.5,  'solid fill'),
    ('list',   1.5,  'simple pattern'),
    ('large',  3.5,  'full pattern'),
]

spacing = 180
canvas.width = len(tiers) * spacing + 40
canvas.height = 220

for i, (size, scale, desc) in enumerate(tiers):
    cx = 80 + i * spacing
    cy = 90
    pid = f"queen_{size}_{i}"

    # Draw backdrop circle scaled to piece size
    r = scale * 25
    canvas.adjust(f"bg_{i}",
        f'<circle cx="{cx}" cy="{cy}" r="{r + 8}" fill="white" stroke="{flag.comp}" stroke-width="1.5"/>')

    # Draw the queen using draw_piece
    flag.draw_piece(
        PieceType.ROOK, MapCord(cx, cy), canvas,
        scale=scale, size=size, piece_id=pid, layer=f"queen_{i}",
    )

    # Labels
    lbl = flag.labelStyle(f"lbl_{i}")
    canvas.add_style(lbl)
    canvas.adjust(f"label_{i}",
        f'<text x="{cx}" y="{cy + r + 30}" text-anchor="middle" '
        f'font-size="13" font-family="\'Cinzel\', sans-serif" class="{lbl.name}">'
        f'{size} (×{scale})</text>'
        f'<text x="{cx}" y="{cy + r + 48}" text-anchor="middle" '
        f'font-size="11" font-family="sans-serif" fill="#666">{desc}</text>')

# Title
canvas.adjust("title",
    f'<text x="{canvas.width//2}" y="20" text-anchor="middle" '
    f'font-size="16" font-family="\'Cinzel\', sans-serif" fill="{flag.darkPrimary}">'
    f'{flag.name} — Queen at 3 tiers (pattern #{flag.patternIndex})</text>')

canvas.show()

So we want to be able to compute yeilds of food per turn. I am thinking of 8 tiers.

I want crop yield to be a combination of flow, temperature, and possibly soil.

we are going to want to create a dotted overlay base upon yeild similar to our climate overlay.

lets work throught the design

I think the temerature and flow are relatively easy to get. Is soil going to make a difference? should we keep it?

Sounds great. so is there a way to calculate how much food you can grow on a tile?

How much of this can you build? I think we need to experiment a bit.

In [ ]:


@dataclass
class FoodYield:
    """Compute per-hex food yield tiers from temperature, water, and soil."""
    terrain: object  # Terrain
    basins: object   # DrainageBasins
    n_tiers: int = 8
    
    # Climate zone index → temperature factor (tune these!)
    temp_curve: dict = field(default_factory=lambda: {
        0: 0.0,   # Ocean/ice
        1: 0.15,  # Tundra
        2: 0.4,   # Boreal
        3: 0.8,   # Temperate
        4: 1.0,   # Subtropical (peak)
        5: 0.6,   # Tropical/hot
    })
    
    # Soil type index → fertility multiplier
    soil_mult: list = field(default_factory=lambda: [
        0.2,  # Granite
        0.3,  # Basalt
        0.7,  # Limestone
        0.6,  # Sandstone
        1.0,  # Alluvial
    ])
    
    def compute(self) -> np.ndarray:
        """Returns per-hex yield as float 0–1, and stores tier (0..n_tiers-1)."""
        t = self.terrain
        n = len(t.elevations)
        
        # --- Temperature factor ---
        climate = t.fields.get('climate', t.compute_climate())
        temp_f = np.array([self.temp_curve.get(int(c), 0.0) for c in climate])
        
        # --- Water factor (precip + flow, diminishing returns) ---
        precip = t.fields.get('precipitation', np.zeros(n))
        
        # Ensure flow field exists
        if 'flow' not in t.fields:
            all_flows = {}
            for ws in self.basins.sheds:
                for idx, fl in ws.tributary._calculate_flow().items():
                    all_flows[idx] = all_flows.get(idx, 0) + fl
            t.fields['flow'] = np.zeros(n)
            for idx, fl in all_flows.items():
                t.fields['flow'][idx] = fl
        
        flow = t.fields['flow']
        
        # Combine precip + flow, log-scale for diminishing returns
        water_raw = precip / 1000.0 + np.log1p(flow) * 0.3
        water_max = np.percentile(water_raw[water_raw > 0], 95) if np.any(water_raw > 0) else 1.0
        water_f = np.clip(water_raw / water_max, 0, 1.0)
        
        # --- Soil factor ---
        soil_type = t.fields.get('soil_type', np.zeros(n, dtype=int))
        soil_f = np.array([self.soil_mult[min(int(s), len(self.soil_mult)-1)] for s in soil_type])
        
        # --- Combine ---
        raw = temp_f * water_f * soil_f
        
        # Zero out ocean
        raw[t.elevations <= 0] = 0.0
        
        # Normalize to 0–1
        rmax = np.percentile(raw[raw > 0], 95) if np.any(raw > 0) else 1.0
        normalized = raw / rmax
        
        # Store
        self.raw = raw
        self.normalized = normalized
        self.tiers = np.clip((normalized * self.n_tiers).astype(int), 0, self.n_tiers - 1)
        t.fields['food_yield'] = self.tiers
        
        return normalized
    
    def summary(self):
        """Print tier distribution."""
        land = self.tiers[self.terrain.elevations > 0]
        print(f"Land hexes: {len(land)}")
        for tier in range(self.n_tiers):
            count = np.sum(land == tier)
            bar = '█' * (count // 10)
            print(f"  Tier {tier}: {count:5d} {bar}")


In [ ]:
# Make sure soil exists
from HexMagic.water.soil import SoilSystem
soil = SoilSystem.from_plates(myTerr, [])

fy = FoodYield(myTerr, basins)
fy.compute()
fy.summary()


I think this is ok. there is a bunch of mountains and oceans

In [ ]:
fy = FoodYield(myTerr, basins)
fy.compute()
fy.summary()

Thoughts

lets build the overlay. I know we need terrainPatterns for the dots. What would be a good color to use. I am assuming we will use it either with a cream backgroun or the elevations. does it make sense to have different colors or just different densities

In [ ]:
from HexMagic.terrainpatterns import TerrainPatterns

def food_overlay(fy: FoodYield, color: str = "#558B2F") -> str:
    """Build dotted overlay from computed food yield tiers."""
    terrain = fy.terrain
    grid = terrain.hexGrid
    
    patGen = TerrainPatterns(terrain)
    patterns = patGen.ballDensity(
        levels=fy.n_tiers,
        fills=[color],
        prefix="food_yield"
    )
    
    overlay = ""
    used = set()
    
    for i in range(len(terrain.elevations)):
        if terrain.elevations[i] <= 0:
            continue
        tier = int(fy.tiers[i])
        if tier == 0:
            continue  # Skip barren — let base map show through
        
        used.add(tier)
        pat_name = patterns[tier].attributes['id']
        hex_obj = grid.hexes[i]
        
        pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
        overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
    
    for tier in sorted(used):
        grid.builder.add_definition(patterns[tier])
    
    return overlay


In [ ]:
svg = food_overlay(fy)
myTerr.hexGrid.builder.layers = []
myTerr.terrainCream()
myTerr.hexGrid.builder.adjust("food", svg)
show(myTerr.hexGrid.builder)


I should make patterns unique


# TerrainDisplay Overlay System

A declarative, composable system for rendering hex terrain with layered overlays. Inspired by FastHTML's component patterns — positional args are overlays, keyword args are context and HTML attributes.

## Quick Start

```python
from HexMagic.overlay import TerrainDisplay, CreamOverlay, RiverOverlay, ClimateOverlay

TerrainDisplay(
    CreamOverlay(),
    RiverOverlay(),
    ClimateOverlay(),
    terrain=my_terrain,
    basins=my_basins,
    id="map"
)
```

That's it. Each overlay is a function that returns an `OverlaySpec`. `TerrainDisplay` renders them bottom-to-top by priority, wraps the result in a `Div(HexTouchMap(...))`, and passes through any HTML/HTMX attributes.

---

## How It Works

### The Three-Phase Render Pipeline

1. **Phase 1 — Run renderers**: Each overlay's `render(ctx)` runs in priority order. Overlays may mutate hex styles (fills, strokes) and/or return an SVG string.
2. **Phase 2 — Bake hex styles**: `grid.update()` is called once, flushing all hex style mutations into the `hexes` layer.
3. **Phase 3 — Stack SVG layers**: Any non-empty SVG strings are added as named layers on top of the hexes, in priority order.

### The Contract

Overlay renderers **may**:
- Mutate hex styles on `ctx.terrain` / `ctx.grid` (e.g. setting `grid.hexes[i].style`)
- Call `ctx.builder.add_definition()`, `ctx.builder.add_style()`, `ctx.builder.add_font()`
- Return an SVG string (or `""` for pure style mutations)

Overlay renderers **must not**:
- Call `ctx.builder.adjust()` — only `TerrainDisplay` does that

This keeps the builder under single ownership and makes overlay ordering predictable.

---

## Available Overlays

| Overlay | Priority | Type | Description |
|---------|----------|------|-------------|
| `TerrainOverlay()` | 5 | hex style | Elevation color bands (green→orange→red) |
| `CreamOverlay()` | 10 | hex style | Parchment tones by elevation + coast distance |
| `SoilOverlay()` | 15 | hex style | Hatch patterns by bedrock type (granite, basalt, etc.) |
| `ClimateOverlay(levels, min_density)` | 40 | SVG | Colored dots by climate zone + precipitation |
| `TemperatureOverlay(ocean_color)` | 41 | SVG | Temperature gradient icons |
| `FlowOverlay()` | 42 | SVG | Drainage flow direction arrows |
| `PrecipitationOverlay(levels, min_density, color)` | 43 | SVG | Rainfall density dots |
| `RiverOverlay(top_n, simplify_k, max_width)` | 60 | SVG | River network from drainage basins |

### Priority = Z-Order

Lower priority → drawn first → at the bottom. Higher priority → drawn last → on top. Think of it as a painter's algorithm: base coat first, fine details last.

---

## Composing Overlays

### Pick and choose

```python
# Minimal — just the parchment base
TerrainDisplay(CreamOverlay(), terrain=t)

# Add rivers (needs basins)
TerrainDisplay(CreamOverlay(), RiverOverlay(), terrain=t, basins=basins)

# Full detail
TerrainDisplay(
    CreamOverlay(),
    ClimateOverlay(levels=4),
    TemperatureOverlay(),
    RiverOverlay(top_n=12),
    terrain=t, basins=basins
)
```

### Order doesn't matter in the call

Priority controls render order, not argument position. These are equivalent:

```python
TerrainDisplay(RiverOverlay(), CreamOverlay(), terrain=t, basins=basins)
TerrainDisplay(CreamOverlay(), RiverOverlay(), terrain=t, basins=basins)
```

Both render cream (priority 10) before rivers (priority 60).

### Presets

Bundle common combinations into helper functions:

```python
def TerrainBase(terrain, **kw):
    return TerrainDisplay(CreamOverlay(), terrain=terrain, **kw)

def DetailedMap(terrain, basins, **kw):
    return TerrainDisplay(
        CreamOverlay(), ClimateOverlay(levels=4),
        RiverOverlay(), TemperatureOverlay(),
        terrain=terrain, basins=basins, **kw
    )
```

---

## Context & Dependencies

`TerrainDisplay` builds an `OverlayContext` from keyword args:

| Keyword | Stored in `ctx` | Enables |
|---------|-----------------|---------|
| `terrain=` | `ctx.terrain`, `ctx.grid`, `ctx.builder` | All overlays |
| `basins=` | `ctx.basins` | `RiverOverlay` |
| `soil=` | `ctx.soil` | `SoilOverlay` |
| `result=` | `ctx.result` (also auto-extracts `.basins`, `.soil`) | Overlays needing zoom result |
| `board=` | `ctx.board` | Borders, Names, Settlements |
| `c2f=` | `ctx.c2f` | Coarse-to-fine mapping for zoomed views |

Each `OverlaySpec` declares a `requires` set. If a required key is missing, the overlay is skipped with a warning — no crash.

```python
RiverOverlay()  # requires={'basins'}
# If basins=None, logs: "Skipping rivers: missing {'basins'}"
```

---

## Creating Custom Overlays

Any function that returns an `OverlaySpec` is an overlay:

```python
def HighPeakMarkers(threshold=2000, color="#FF4444", **kw) -> OverlaySpec:
    """Red circles on hexes above a given elevation."""
    def render(ctx: OverlayContext) -> str:
        parts = []
        for i, elev in enumerate(ctx.terrain.elevations):
            if elev > threshold:
                c = ctx.grid.hexes[i].center
                parts.append(f'<circle cx="{c.x}" cy="{c.y}" r="4" fill="{color}" opacity="0.7"/>')
        return '\n'.join(parts)
    return OverlaySpec("high_peaks", render, priority=55)

# Use it
TerrainDisplay(CreamOverlay(), HighPeakMarkers(threshold=1500), terrain=t)
```

### Custom overlay with hex style mutations

Return `""` and mutate styles instead:

```python
def OceanTintOverlay(color="#B3E5FC", **kw) -> OverlaySpec:
    """Recolor all ocean hexes."""
    def render(ctx: OverlayContext) -> str:
        style = StyleCSS("ocean_tint", fill=color)
        ctx.builder.add_style(style)
        for i, elev in enumerate(ctx.terrain.elevations):
            if elev <= 0:
                ctx.grid.hexes[i].style = style
        return ""
    return OverlaySpec("ocean_tint", render, priority=8)
```

### Custom overlay with dependencies

```python
def BasinLabelOverlay(**kw) -> OverlaySpec:
    """Label each drainage basin at its mouth."""
    def render(ctx: OverlayContext) -> str:
        # ctx.basins is guaranteed present because of requires
        labels = []
        for basin in ctx.basins.top_basins(5):
            mouth = ctx.grid.hexes[basin.mouth].center
            labels.append(f'<text x="{mouth.x}" y="{mouth.y}" font-size="10">{basin.name}</text>')
        return '\n'.join(labels)
    return OverlaySpec("basin_labels", render, requires={'basins'}, priority=65)
```

---

## HTMX & HTML Attributes

All `**kwargs` that aren't recognized context keys pass through to the outer `Div`:

```python
TerrainDisplay(
    CreamOverlay(), RiverOverlay(),
    terrain=t, basins=basins,
    id="map",
    cls="w-full h-full",
    hx_swap="outerHTML",
    hx_get="/refresh_map"
)
# Produces: Div(HexTouchMap(...), id="map", cls="w-full h-full", hx_swap="outerHTML", ...)
```

Use `map_cls=` to set the class on the inner `HexTouchMap` (default: `"w-full h-full"`).

---

## Debug Mode

Pass `debug=True` to get the builder's diagnostic view instead of the rendered map:

```python
TerrainDisplay(CreamOverlay(), SoilOverlay(), terrain=t, soil=soil, debug=True)
```

This shows definitions, styles, and layers with their sizes — useful for checking that overlays registered correctly without side-effecting the builder.

---

## Route Integration Example

```python
@rt("/settlement_map/{id}")
def settlement_map(session, id: str):
    active = get_active_game(session)
    settlement = get_settlement(id)
    result = active.cover.zoom_region_fast(settlement.region(active.grid, rings=5))

    overlays = session_overlays(session)  # e.g. {'cream', 'rivers', 'climate'}
    components = overlay_set_to_components(overlays)

    return TerrainDisplay(
        *components,
        terrain=result.terrain,
        result=result,
        board=active.board,
        c2f=result.c2f,
        id="map",
        hx_swap="outerHTML"
    )

def overlay_set_to_components(names: set) -> list:
    """Convert a set of overlay names to component instances."""
    registry = {
        'cream': CreamOverlay,
        'soil': SoilOverlay,
        'climate': ClimateOverlay,
        'temperature': TemperatureOverlay,
        'rivers': RiverOverlay,
        'flow': FlowOverlay,
        'precipitation': PrecipitationOverlay,
    }
    return [registry[k]() for k in names if k in registry]
```

---

## Summary

| Concept | Pattern |
|---------|---------|
| Add an overlay | Pass `SomeOverlay()` as a positional arg |
| Configure an overlay | Pass params: `ClimateOverlay(levels=4)` |
| Set render order | Set `priority` in `OverlaySpec` (low = bottom) |
| Declare dependencies | Set `requires={'basins'}` in `OverlaySpec` |
| Pass HTML attrs | Use keyword args: `id=`, `cls=`, `hx_swap=` |
| Debug | `debug=True` |
| Create custom | Write a function returning `OverlaySpec` |
```

Lets make a FoodOverlay in the new overlayStyle system

In [ ]:
def FoodOverlay(color: str = "#558B2F", n_tiers: int = 8, skip_zero: bool = True, **kw):
    """Dotted density overlay showing food yield per hex."""
    
    def render(ctx):
        fy = FoodYield(ctx.terrain, ctx.basins, n_tiers=n_tiers)
        fy.compute()
        
        grid = ctx.grid
        patGen = TerrainPatterns(ctx.terrain)
        patterns = patGen.ballDensity(
            levels=n_tiers,
            fills=[color],
            prefix="food_yield"
        )
        
        overlay = ""
        used = set()
        
        for i in range(len(ctx.terrain.elevations)):
            if ctx.terrain.elevations[i] <= 0:
                continue
            tier = int(fy.tiers[i])
            if skip_zero and tier == 0:
                continue
            
            used.add(tier)
            pat_name = patterns[tier].attributes['id']
            hex_obj = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
        
        for tier in sorted(used):
            ctx.builder.add_definition(patterns[tier])
        
        return overlay
    
    return OverlaySpec("food_yield", render, requires={'basins'}, priority=44)


TerrainDisplay(
    CreamOverlay(),
    FoodOverlay(color="#8D6E63", n_tiers=6),
    RiverOverlay(),
    terrain=myTerr,
    basins=basins,
)


In [ ]:
!cat ../../HexMagic/game/piece.py

So we now have the basics to have food on the map we should work out the mechanics of how pieces move to food harvest and share food with one another. 

1. we need a dystyra path to pick a target
2. something that will assign a piece
3. the share mechanism. my idea for this is the same a the context where they have a cone coming out from them a sight modified number of hexes. feeding priority is given by rank and then who has the least amount of food. you will always keep 4 turns of your diet of food so you won't starve yourself.

Other things we should add?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good question — here are a few mechanics that would naturally complement what you've got:

- **Consumption rate per turn** — each piece eats X food/turn (probably varies by piece type). Without this the harvest/share loop has no pressure.
- **Starvation penalty** — when food hits 0, what happens? Health drain? Movement debuff? Both? This drives the urgency to path toward food.
- **Tile depletion & regrowth** — does harvesting reduce a tile's yield for N turns? If so, you get natural migration pressure and competition between kingdoms.
- **Settlement granaries** — settlements could stockpile surplus beyond what any one piece carries, acting as a "food bank" pieces resupply from. This ties nicely into your existing Settlement model.
- **Seasonal variation** — if your turn clock maps to seasons, you could scale food yields (e.g. winter = 0.3× temperate yield), which creates stockpiling strategy.

Which of these feel most important to get right first? I'd suggest **consumption + starvation** since that's what makes the whole food system actually matter mechanically — without it, harvesting is just decoration.

In [ ]:
??Piece

So we have food, diet and food_capacity on pieces. If starving you start to loose health. all health lost you die

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good — the core loop is clean: each turn `food -= diet`, and if food is 0 at consumption time, health takes damage instead.

A couple of design questions before we wire it up:

**Starvation damage rate** — how harsh should it be? Options:
- **Fixed** (e.g. lose 10 HP/turn while starving) — simple, predictable
- **Escalating** (e.g. 5 first turn, 10 next, 20 next…) — creates urgency curve
- **Diet-proportional** (e.g. lose `diet × 10` HP) — bigger units starve faster

Which feels right for your game's pacing?

Diet-proportional

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice — so the per-turn logic would be:

1. Try to eat: `shortfall = max(0, diet - food)`
2. Consume what's available: `food = max(0, food - diet)`
3. If `shortfall > 0`: `health -= shortfall * STARVE_MULT`
4. If `health <= 0`: piece dies

`STARVE_MULT` is your tuning knob — it converts missed food into HP damage. With `diet=1.0` and `STARVE_MULT=10`, a fully starving pawn loses 10 HP/turn (dead in 10 turns). A knight with `diet=2.0` loses 20 HP/turn (dead in 5).

Does 10 feel like the right multiplier, or do you want starvation to be faster/slower?

Lets do 5, I am a generous king.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



`STARVE_MULT = 5` it is — a pawn (`diet=1.0`) starves in 20 turns, a knight (`diet=2.0`) in 10. Generous but still consequential.

Now for the **harvest** side — when a piece does a HARVEST action on a tile, how much food do they collect? Two obvious options:

- **`harvest_strength × tile_tier`** — strong harvesters on rich land get a lot
- **`harvest_strength × tile_yield_normalized`** — continuous scale (0–1) instead of discrete tiers

The first is simpler and maps directly onto your 8-tier `FoodYield`. Something like `food_gained = harvest_strength * tier` — so a pawn (harvest=1) on tier 5 land gets 5 food, while a bishop (harvest=3) gets 15.

Which approach do you prefer?

harvest_strength × tile_tier

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



Great — `food_gained = harvest_strength × tier`. Simple, intuitive, and the tier numbers do double duty as both visual density and gameplay value.

So the full per-turn food cycle for a piece is:

1. **Harvest** (if HARVEST instruction): `food += harvest_strength × tier` (capped at `food_capacity`)
2. **Share** (if GIVE instruction): distribute from cone — we'll design this next
3. **Eat**: `shortfall = max(0, diet - food)` → `food -= diet` → `health -= shortfall * 5`
4. **Death check**: `health <= 0` → remove piece

Ready to tackle the **sharing cone** mechanic? You mentioned sight-modified range, rank priority, then lowest-food-first, and keeping 4 turns of diet as a reserve floor.

In [ ]:
!cat ../../HexMagic/plot/*.py

So we have quite a bit of math that uses hexpostions to find adjacent hexes

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Exactly — you've already got `hex_in_cone` and `directions_in_cone` in `cube.py`, which is perfect for the share cone. The GIVE mechanic would basically be:

1. Get facing direction: `HexPosition.directions()[piece.facing]`
2. Get all hexes in range: `grid.indices_in_range(piece.location, piece.sight)`
3. Filter with `hex_in_cone(relative_pos, facing_dir, allowed_rotations=1)` to get the cone
4. Find other pieces in those hexes
5. Sort receivers by rank (piece_type), then lowest food first
6. Distribute surplus (keeping `4 × diet` reserve for self)

How wide should the cone be? `allowed_rotations=1` gives a 3-direction wedge (120°), `allowed_rotations=2` gives 5 directions (300° — almost everything). Which feels right?

allowed_rotations=1

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good — 120° wedge, clean and directional. So the GIVE step is:

1. Cone = `sight` range, `facing ± 1` direction
2. Find allied pieces in cone
3. Surplus = `food - 4 × diet` (reserve 4 turns for self)
4. Sort receivers: **rank first**, then lowest food
5. Fill each receiver toward their capacity until surplus runs out

One thing we need to nail down: **what's the rank order?** Which piece types get fed first? Something like KING > QUEEN > ROOK > BISHOP > KNIGHT > PAWN, or do you have a different hierarchy in mind?

 KING > QUEEN > ROOK > BISHOP > KNIGHT > PAWN

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Perfect. So the full GIVE algorithm per piece per turn:

1. **Cone**: hexes within `sight` range, facing ± 1 rotation
2. **Surplus**: `max(0, food - 4 × diet)`
3. **Receivers**: allied pieces in cone, sorted by `KING > QUEEN > ROOK > BISHOP > KNIGHT > PAWN`, then lowest `food` first within same rank
4. **Distribute**: for each receiver, give `min(surplus, receiver.food_capacity - receiver.food)`, decrement surplus

One edge case — should a piece give food to a piece that **also** has a GIVE instruction this turn? Or does the whole sharing phase resolve simultaneously (snapshot food levels, then apply all transfers)?

In theory each piece does its thing in isolation. We are going to build rules a level up, but at the piece level if told to give you give.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Clean — each piece resolves its own instruction independently. No snapshot needed, just execute in whatever order the game loop processes pieces.

So the full per-turn cycle for a single piece is now:

1. **Execute instruction** (FORWARD / HARVEST / GIVE / DEFEND / PAUSE / SETTLE / ROT)
2. **Eat**: `shortfall = max(0, diet - food)` → `food = max(0, food - diet)` → `health -= shortfall × 5`
3. **Death**: if `health ≤ 0`, remove piece

With HARVEST doing `food += min(harvest_strength × tier, food_capacity - food)` and GIVE doing the cone distribution.

Ready to start coding this up? Want me to sketch the `give` method, or would you rather build it yourself?

How much of this can you build?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Pretty much all of it. Here's the core food mechanics module:

```python
#| export
from HexMagic.plot.cube import HexPosition, hex_in_cone

STARVE_MULT = 5
FOOD_RESERVE_TURNS = 4

# Lower index = higher priority (fed first)
RANK_ORDER = {
    PieceType.KING:   0,
    PieceType.QUEEN:  1,
    PieceType.ROOK:   2,
    PieceType.BISHOP: 3,
    PieceType.KNIGHT: 4,
    PieceType.PAWN:   5,
}


def harvest(piece: Piece, grid: HexGrid, food_tiers: np.ndarray):
    """HARVEST action: gather food from current tile."""
    if piece.location is None or piece.location < 0:
        return
    tier = int(food_tiers[piece.location])
    gained = piece.harvest_strength * tier
    piece.food = min(piece.food + gained, piece.food_capacity)


def give(piece: Piece, grid: HexGrid, all_pieces: list[Piece]):
    """GIVE action: distribute surplus food to allies in facing cone."""
    reserve = FOOD_RESERVE_TURNS * piece.diet
    surplus = max(0.0, piece.food - reserve)
    if surplus <= 0:
        return

    facing_dir = HexPosition.directions()[piece.facing % 6]

    # Find allied pieces in cone
    receivers = []
    for other in all_pieces:
        if other is piece or other.location is None:
            continue
        if other.owner_id != piece.owner_id:
            continue
        if other.food >= other.food_capacity:
            continue

        dist = grid.index_to_hexposition(other.location, piece.location)
        if abs(dist) > piece.sight:
            continue
        if not hex_in_cone(dist, facing_dir, allowed_rotations=1):
            continue

        receivers.append(other)

    # Sort: rank first (lower = higher priority), then lowest food
    receivers.sort(key=lambda p: (RANK_ORDER.get(p.piece_type, 99), p.food))

    # Distribute
    for receiver in receivers:
        if surplus <= 0:
            break
        room = receiver.food_capacity - receiver.food
        amount = min(surplus, room)
        receiver.food += amount
        piece.food -= amount
        surplus -= amount


def eat(piece: Piece) -> bool:
    """Consume food; apply starvation damage. Returns True if alive."""
    shortfall = max(0.0, piece.diet - piece.food)
    piece.food = max(0.0, piece.food - piece.diet)
    if shortfall > 0:
        piece.health -= shortfall * STARVE_MULT
    return piece.health > 0


def piece_food_step(piece: Piece, instruction: 'Instruction',
                    grid: HexGrid, food_tiers: np.ndarray,
                    all_pieces: list[Piece]) -> bool:
    """Run one turn's food cycle for a single piece.
    
    Call AFTER movement has been resolved.
    Returns True if piece is still alive.
    """
    if instruction == Instruction.HARVEST:
        harvest(piece, grid, food_tiers)
    elif instruction == Instruction.GIVE:
        give(piece, grid, all_pieces)

    return eat(piece)
```

This gives you:
- **`harvest()`** — `harvest_strength × tier`, capped at capacity
- **`give()`** — cone within `sight`, rank then lowest-food priority, keeps 4-turn reserve
- **`eat()`** — consumes `diet`, starvation at `5×` shortfall, returns alive/dead
- **`piece_food_step()`** — orchestrator you call per piece per turn after movement resolves

Want to test it on your Japan/Korea map, or tweak any of the numbers first?

lets extract out a pieces in sight with a multipler (pieces at higher elevations are going to see farther, but we can figure out by home much by later). it gives us a nice board balance where the higher ground has strategic advantage, but the lower ground has more food.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



Great instinct — elevation advantage for sight is a classic tradeoff. Here's the extraction:

```python
#| export

def pieces_in_sight(piece: Piece, grid: HexGrid, elevations: np.ndarray,
                    all_pieces: list[Piece],
                    elevation_mult: float = 0.005,
                    facing_only: bool = False,
                    allowed_rotations: int = 1,
                    allied_only: bool = False) -> list[Piece]:
    """Find pieces visible from piece's location.
    
    Effective sight = piece.sight + elevation * elevation_mult
    
    Args:
        piece: The observing piece
        grid: HexGrid
        elevations: Terrain elevation array
        all_pieces: All pieces on the board
        elevation_mult: Sight bonus per unit of elevation (tune later)
        facing_only: If True, restrict to facing cone
        allowed_rotations: Cone width (1 = 120°)
        allied_only: If True, only return same owner_id
    
    Returns:
        List of visible pieces (excludes self)
    """
    if piece.location is None or piece.location < 0:
        return []
    
    elev = max(0, elevations[piece.location])
    effective_sight = piece.sight + elev * elevation_mult
    
    facing_dir = HexPosition.directions()[piece.facing % 6]
    
    visible = []
    for other in all_pieces:
        if other is piece or other.location is None:
            continue
        if allied_only and other.owner_id != piece.owner_id:
            continue
        
        relative = grid.index_to_hexposition(other.location, piece.location)
        dist = abs(relative)
        if dist > effective_sight:
            continue
        
        if facing_only and not hex_in_cone(relative, facing_dir, allowed_rotations):
            continue
        
        visible.append(other)
    
    return visible
```

Then `give()` simplifies to:

```python
#| export

def give(piece: Piece, grid: HexGrid, elevations: np.ndarray,
         all_pieces: list[Piece]):
    """GIVE action: distribute surplus food to allies in facing cone."""
    reserve = FOOD_RESERVE_TURNS * piece.diet
    surplus = max(0.0, piece.food - reserve)
    if surplus <= 0:
        return

    receivers = [
        p for p in pieces_in_sight(
            piece, grid, elevations, all_pieces,
            facing_only=True, allowed_rotations=1, allied_only=True
        )
        if p.food < p.food_capacity
    ]

    receivers.sort(key=lambda p: (RANK_ORDER.get(p.piece_type, 99), p.food))

    for receiver in receivers:
        if surplus <= 0:
            break
        amount = min(surplus, receiver.food_capacity - receiver.food)
        receiver.food += amount
        piece.food -= amount
        surplus -= amount
```

Now `pieces_in_sight` is reusable for combat targeting, scouting, alerts — anything where "who can I see?" matters. The `elevation_mult` of `0.005` means a hex at elevation 200 gets +1 sight range. We can tune that once you see it on the board.

We now have
```
def field_of_view(origin: HexPosition, facing: HexPosition, max_distance: int) -> list[HexPosition]:
    """Get all hexes within a 120° field of view (±60° from facing) up to max_distance."""
    results = []
    for radius in range(1, max_distance + 1):
        for hex_pos in origin.ring(radius):
            if hex_in_cone(hex_pos - origin, facing, 1):
                results.append(hex_pos)
    return results
```

In [ ]:
#| export
from HexMagic.plot.cube import HexPosition, field_of_view

STARVE_MULT = 5
FOOD_RESERVE_TURNS = 4

# Lower index = higher priority (fed first)
RANK_ORDER = {
    PieceType.KING:   0,
    PieceType.QUEEN:  1,
    PieceType.ROOK:   2,
    PieceType.BISHOP: 3,
    PieceType.KNIGHT: 4,
    PieceType.PAWN:   5,
}


def harvest(piece: Piece, grid: HexGrid, food_tiers: np.ndarray):
    """HARVEST action: gather food from current tile."""
    if piece.location is None or piece.location < 0:
        return
    tier = int(food_tiers[piece.location])
    gained = piece.harvest_strength * tier
    piece.food = min(piece.food + gained, piece.food_capacity)



def eat(piece: Piece) -> bool:
    """Consume food; apply starvation damage. Returns True if alive."""
    shortfall = max(0.0, piece.diet - piece.food)
    piece.food = max(0.0, piece.food - piece.diet)
    if shortfall > 0:
        piece.health -= shortfall * STARVE_MULT
    return piece.health > 0


def piece_food_step(piece: Piece, instruction: 'Instruction',
                    grid: HexGrid, food_tiers: np.ndarray,
                    all_pieces: list[Piece]) -> bool:
    """Run one turn's food cycle for a single piece.
    
    Call AFTER movement has been resolved.
    Returns True if piece is still alive.
    """
    if instruction == Instruction.HARVEST:
        harvest(piece, grid, food_tiers)
    elif instruction == Instruction.GIVE:
        give(piece, grid, all_pieces)

    return eat(piece)


In [ ]:
@patch
def pieces_in_sight(piece: Piece, grid: HexGrid, elevations: np.ndarray,
                    all_pieces: list[Piece],
                    elevation_mult: float = 0.005,
                    facing_only: bool = False,
                    allied_only: bool = False) -> list[Piece]:
    """Find pieces visible from piece's location.
    
    Effective sight = piece.sight + elevation * elevation_mult
    facing_only restricts to 120° FOV cone (uses field_of_view).
    """
    if piece.location is None or piece.location < 0:
        return []

    elev = max(0, elevations[piece.location])
    effective_sight = int(piece.sight + elev * elevation_mult)
    facing_dir = HexPosition.directions()[piece.facing % 6]

    # Pre-build FOV set — O(1) lookup per piece
    if facing_only:
        fov_set = set(field_of_view(HexPosition.origin(), facing_dir, effective_sight))

    visible = []
    for other in all_pieces:
        if other is piece or other.location is None:
            continue
        if allied_only and other.owner_id != piece.owner_id:
            continue

        relative = grid.index_to_hexposition(other.location, piece.location)

        if facing_only:
            if relative not in fov_set:
                continue
        else:
            if abs(relative) > effective_sight:
                continue

        visible.append(other)

    return visible


In [ ]:
??hex_in_cone

In [ ]:
directions_in_cone??

In [ ]:
@patch
def give(piece: Piece, grid: HexGrid, elevations: np.ndarray,
         all_pieces: list[Piece]):
    """GIVE action: distribute surplus food to allies in facing cone."""
    reserve = FOOD_RESERVE_TURNS * piece.diet
    surplus = max(0.0, piece.food - reserve)
    if surplus <= 0:
        return

    receivers = [
        p for p in piece.pieces_in_sight(
            elevations, all_pieces,
            facing_only=True, allowed_rotations=1, allied_only=True
        )
        if p.food < p.food_capacity
    ]

    receivers.sort(key=lambda p: (RANK_ORDER.get(p.piece_type, 99), p.food))

    for receiver in receivers:
        if surplus <= 0:
            break
        amount = min(surplus, receiver.food_capacity - receiver.food)
        receiver.food += amount
        piece.food -= amount
        surplus -= amount

can we switch things to field_of_view

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes — and it actually solves the `hex_in_cone` import issue too, since `field_of_view` is already working. The key insight is: pre-compute the FOV as a set of valid relative positions, then just do membership checks:

```python
@patch
def pieces_in_sight(piece: Piece, grid: HexGrid, elevations: np.ndarray,
                    all_pieces: list[Piece],
                    elevation_mult: float = 0.005,
                    facing_only: bool = False,
                    allied_only: bool = False) -> list[Piece]:
    """Find pieces visible from piece's location.
    
    Effective sight = piece.sight + elevation * elevation_mult
    facing_only restricts to 120° FOV cone (uses field_of_view).
    """
    if piece.location is None or piece.location < 0:
        return []

    elev = max(0, elevations[piece.location])
    effective_sight = int(piece.sight + elev * elevation_mult)
    facing_dir = HexPosition.directions()[piece.facing % 6]

    # Pre-build FOV set — O(1) lookup per piece
    if facing_only:
        fov_set = set(field_of_view(HexPosition.origin(), facing_dir, effective_sight))

    visible = []
    for other in all_pieces:
        if other is piece or other.location is None:
            continue
        if allied_only and other.owner_id != piece.owner_id:
            continue

        relative = grid.index_to_hexposition(other.location, piece.location)

        if facing_only:
            if relative not in fov_set:
                continue
        else:
            if abs(relative) > effective_sight:
                continue

        visible.append(other)

    return visible
```

The `give` function stays the same — it already calls `pieces_in_sight` with `facing_only=True`. The `allowed_rotations` parameter can be dropped since `field_of_view` hardcodes 120° which is exactly what we want.

Can you rewrite this using the @patch style?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
#| export
import heapq

@patch
def pathfind(self: Piece, target: int, grid: HexGrid,
             elevations: np.ndarray,
             countries: np.ndarray = None) -> list[int]:
    """Dijkstra from piece.location to target hex, respecting movement costs.
    
    Returns list of hex indices (start → target), or [] if unreachable.
    """
    start = self.location
    if start is None or start < 0:
        return []
    
    pq = [(0.0, start, [start])]
    visited = set()
    
    while pq:
        cost, current, path = heapq.heappop(pq)
        
        if current == target:
            return path
        
        if current in visited:
            continue
        visited.add(current)
        
        for neighbor in grid.neighborsOf(current):
            if neighbor in visited:
                continue
            if neighbor in grid.invalidRegion:
                continue
            if elevations[neighbor] <= 0:
                continue
            if countries is not None and countries[neighbor] < 0:
                continue
            
            edge_cost = _move_cost(elevations, current, neighbor)
            heapq.heappush(pq, (cost + edge_cost, neighbor, path + [neighbor]))
    
    return []


@patch
def pathfind_to_rules(self: Piece, target: int, grid: HexGrid,
                      elevations: np.ndarray,
                      countries: np.ndarray = None) -> InstructionList:
    """Find path to target and convert to movement instructions."""
    path = self.pathfind(target, grid, elevations, countries)
    if len(path) < 2:
        return InstructionList()
    
    rules = InstructionList.path_to_rules(path, grid, start_facing=self.facing)
    return InstructionList(rules, cursor=0, patrol=False)


@patch
def pathfind_to_food(self: Piece, grid: HexGrid,
                     elevations: np.ndarray, food_tiers: np.ndarray,
                     min_tier: int = 3,
                     countries: np.ndarray = None) -> InstructionList:
    """Find nearest tile with food >= min_tier and path there + harvest."""
    start = self.location
    if start is None or start < 0:
        return InstructionList()
    
    pq = [(0.0, start)]
    visited = set()
    parent = {start: None}
    
    while pq:
        cost, current = heapq.heappop(pq)
        
        if current in visited:
            continue
        visited.add(current)
        
        if current != start and int(food_tiers[current]) >= min_tier:
            # Reconstruct path
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = parent[node]
            path.reverse()
            
            rules = InstructionList.path_to_rules(path, grid, start_facing=self.facing)
            rules.append(Instruction.HARVEST.value)
            return InstructionList(rules, cursor=0, patrol=False)
        
        for neighbor in grid.neighborsOf(current):
            if neighbor in visited or neighbor in grid.invalidRegion:
                continue
            if elevations[neighbor] <= 0:
                continue
            if countries is not None and countries[neighbor] < 0:
                continue
            
            edge_cost = _move_cost(elevations, current, neighbor)
            if neighbor not in parent:
                parent[neighbor] = current
                heapq.heappush(pq, (cost + edge_cost, neighbor))
    
    return InstructionList()
```

Now it's `piece.pathfind(target, grid, elevations)`, `piece.pathfind_to_rules(target, grid, elevations)`, and `piece.pathfind_to_food(grid, elevations, food_tiers)` — consistent with the rest of your Piece API.

In [ ]:

@patch
def pathfind(self: Piece, target: int, grid: HexGrid,
             elevations: np.ndarray,
             countries: np.ndarray = None) -> list[int]:
    """Dijkstra from piece.location to target hex, respecting movement costs.
    
    Returns list of hex indices (start → target), or [] if unreachable.
    """
    start = self.location
    if start is None or start < 0:
        return []
    
    pq = [(0.0, start, [start])]
    visited = set()
    
    while pq:
        cost, current, path = heapq.heappop(pq)
        
        if current == target:
            return path
        
        if current in visited:
            continue
        visited.add(current)
        
        for neighbor in grid.neighborsOf(current):
            if neighbor in visited:
                continue
            if neighbor in grid.invalidRegion:
                continue
            if elevations[neighbor] <= 0:
                continue
            if countries is not None and countries[neighbor] < 0:
                continue
            
            edge_cost = _move_cost(elevations, current, neighbor)
            heapq.heappush(pq, (cost + edge_cost, neighbor, path + [neighbor]))
    
    return []


@patch
def pathfind_to_rules(self: Piece, target: int, grid: HexGrid,
                      elevations: np.ndarray,
                      countries: np.ndarray = None) -> InstructionList:
    """Find path to target and convert to movement instructions."""
    path = self.pathfind(target, grid, elevations, countries)
    if len(path) < 2:
        return InstructionList()
    
    rules = InstructionList.path_to_rules(path, grid, start_facing=self.facing)
    return InstructionList(rules, cursor=0, patrol=False)


@patch
def pathfind_to_food(self: Piece, grid: HexGrid,
                     elevations: np.ndarray, food_tiers: np.ndarray,
                     min_tier: int = 3,
                     countries: np.ndarray = None) -> InstructionList:
    """Find nearest tile with food >= min_tier and path there + harvest."""
    start = self.location
    if start is None or start < 0:
        return InstructionList()
    
    pq = [(0.0, start)]
    visited = set()
    parent = {start: None}
    
    while pq:
        cost, current = heapq.heappop(pq)
        
        if current in visited:
            continue
        visited.add(current)
        
        if current != start and int(food_tiers[current]) >= min_tier:
            # Reconstruct path
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = parent[node]
            path.reverse()
            
            rules = InstructionList.path_to_rules(path, grid, start_facing=self.facing)
            rules.append(Instruction.HARVEST.value)
            return InstructionList(rules, cursor=0, patrol=False)
        
        for neighbor in grid.neighborsOf(current):
            if neighbor in visited or neighbor in grid.invalidRegion:
                continue
            if elevations[neighbor] <= 0:
                continue
            if countries is not None and countries[neighbor] < 0:
                continue
            
            edge_cost = _move_cost(elevations, current, neighbor)
            if neighbor not in parent:
                parent[neighbor] = current
                heapq.heappush(pq, (cost + edge_cost, neighbor))
    
    return InstructionList()


I think this is so strong. I want to build two things. 1 a sight overlay so we can check a piece's range

but before I want to build a surround command for a given group of pieces and a hex. we can use https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csgraph.maximum_bipartite_matching.html to match pieces to designations and then use our shortest path to give everyone routes. those that have a shorter sight could be in closer.

thoughts on the surround strategy for a hex

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Love this — it's a proper tactical AI command. A few design thoughts:

**Position selection**: Generate candidate surround positions as concentric rings around the target. Ring 1 = 6 hexes, ring 2 = 12, ring 3 = 18. Filter out water/impassable.

**Sight-aware layering**: Short-sight pieces *must* be in inner rings to see the target. Long-sight pieces *can* go further out. So the constraint is: piece at position P must have `effective_sight >= distance(P, target)`.

**Matching**: I'd actually suggest `scipy.optimize.linear_sum_assignment` (Hungarian algorithm) instead of `maximum_bipartite_matching` — it minimizes *total cost*, not just cardinality. Your cost matrix would be:

```
cost[piece_i, position_j] = pathfind_cost(piece_i → position_j)
                           = ∞ if piece can't see target from position_j
```

This naturally puts short-sight pieces close and minimizes total marching distance.

**The algorithm**:
1. Generate rings 1..max_ring around target, filter to valid land hexes
2. For each (piece, position) pair, compute: can piece see target from there? If yes, pathfind cost. If no, ∞.
3. `linear_sum_assignment(cost_matrix)` → optimal piece↔position mapping
4. `piece.pathfind_to_rules(assigned_position, ...)` for each piece

**One question**: should all pieces face *toward* the target when they arrive? That way GIVE cones point inward (supply chain) and sight covers the target. Or do you want facing to be part of the strategy too?

I love Hungarians, I was once taught by Egon Balas. I think we want two option - one facing inward as if they are feeding the queen and one facing outward so they could pretect her.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Egon Balas! The man who made branch-and-cut what it is — what a privilege. He'd appreciate the elegance of using Hungarian matching on a hex grid.

Two modes then:

- **FEED** — face inward toward target. GIVE cones converge on the center. Supply chain formation.
- **GUARD** — face outward away from target. Sight cones scan the perimeter. Defensive formation.

```python
#| export
from scipy.optimize import linear_sum_assignment
from enum import Enum

class SurroundMode(Enum):
    FEED  = "feed"   # face inward — supply chain
    GUARD = "guard"  # face outward — defensive perimeter


@patch
def effective_sight(self: Piece, elevations: np.ndarray,
                    elevation_mult: float = 0.005) -> float:
    """Sight range including elevation bonus."""
    if self.location is None or self.location < 0:
        return float(self.sight)
    elev = max(0, elevations[self.location])
    return self.sight + elev * elevation_mult


def _facing_toward(grid: HexGrid, from_idx: int, to_idx: int) -> int:
    """Direction index (0–5) from from_idx toward to_idx."""
    rel = grid.index_to_hexposition(to_idx, from_idx)
    # Find closest cardinal direction
    dirs = HexPosition.directions()
    best_dir = 0
    best_dot = -999
    for i, d in enumerate(dirs):
        # Dot product in cube coords
        dot = rel.q * d.q + rel.r * d.r + rel.s * d.s
        if dot > best_dot:
            best_dot = dot
            best_dir = i
    return best_dir


def _facing_away(grid: HexGrid, from_idx: int, to_idx: int) -> int:
    """Direction index (0–5) pointing away from to_idx."""
    return (_facing_toward(grid, from_idx, to_idx) + 3) % 6


def surround(pieces: list[Piece], target: int,
             grid: HexGrid, elevations: np.ndarray,
             mode: SurroundMode = SurroundMode.GUARD,
             max_ring: int = 3,
             countries: np.ndarray = None,
             elevation_mult: float = 0.005
             ) -> dict[str, InstructionList]:
    """Assign pieces to surround positions using Hungarian matching.
    
    Args:
        pieces: Pieces to assign (must have locations)
        target: Hex index to surround
        grid: HexGrid
        elevations: Elevation array
        mode: FEED (face inward) or GUARD (face outward)
        max_ring: Maximum ring distance from target
        countries: Country array (optional, for impassable)
        elevation_mult: Sight bonus per elevation unit
    
    Returns:
        Dict mapping piece.id → InstructionList with path + final rotation
    """
    # 1. Generate candidate positions (rings 1..max_ring)
    candidates = []
    for ring in range(1, max_ring + 1):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, target)
            if idx < 0 or idx in grid.invalidRegion:
                continue
            if elevations[idx] <= 0:
                continue
            if countries is not None and countries[idx] < 0:
                continue
            candidates.append((idx, ring))
    
    if not candidates or not pieces:
        return {}
    
    n_pieces = len(pieces)
    n_positions = len(candidates)
    
    # 2. Build cost matrix: pieces × positions
    INF = 1e9
    cost = np.full((n_pieces, n_positions), INF)
    
    for i, piece in enumerate(pieces):
        if piece.location is None or piece.location < 0:
            continue
        
        # Effective sight at current location (conservative — use current elev)
        eff_sight = piece.effective_sight(elevations, elevation_mult)
        
        for j, (pos_idx, ring_dist) in enumerate(candidates):
            # Can piece see target from this position?
            if ring_dist > eff_sight:
                continue  # can't see target from here
            
            # Pathfind cost
            path = piece.pathfind(pos_idx, grid, elevations, countries)
            if not path:
                continue  # unreachable
            
            # Cost = path length (weighted by movement costs)
            path_cost = 0.0
            for k in range(len(path) - 1):
                path_cost += _move_cost(elevations, path[k], path[k + 1])
            
            cost[i, j] = path_cost
    
    # 3. Hungarian matching
    row_ind, col_ind = linear_sum_assignment(cost)
    
    # 4. Generate instruction lists
    assignments = {}
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue  # unmatched
        
        piece = pieces[i]
        pos_idx = candidates[j][0]
        
        path = piece.pathfind(pos_idx, grid, elevations, countries)
        if len(path) < 2:
            continue
        
        rules = InstructionList.path_to_rules(path, grid, start_facing=piece.facing)
        
        # Append final facing rotation
        if mode == SurroundMode.FEED:
            desired_facing = _facing_toward(grid, pos_idx, target)
        else:
            desired_facing = _facing_away(grid, pos_idx, target)
        
        # Calculate rotations needed to reach desired facing
        # Current facing after path (last rule's facing)
        # Simulate to get final facing — or just compute from path end
        path_end_facing = _facing_toward(grid, path[-2], path[-1]) if len(path) >= 2 else piece.facing
        
        # Shortest rotation to desired
        diff = (desired_facing - path_end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))
        
        assignments[piece.id] = InstructionList(rules, cursor=0, patrol=False)
    
    return assignments
```

Usage would be:

```python
orders = surround(my_pawns, queen.location, grid, elevations,
                  mode=SurroundMode.FEED)

for piece in my_pawns:
    if piece.id in orders:
        piece.instructions = orders[piece.id]
```

The `_facing_toward` / `_facing_away` helpers are reusable too — useful any time you need a piece to look at something.

Want to test this on the board, or build the sight overlay first so we can visually verify the cones?

In [ ]:
class SurroundMode(Enum):
    FEED  = "feed"   # face inward — supply chain
    GUARD = "guard"  # face outward — defensive perimeter


@patch
def effective_sight(self: Piece, elevations: np.ndarray,
                    elevation_mult: float = 0.005) -> float:
    """Sight range including elevation bonus."""
    if self.location is None or self.location < 0:
        return float(self.sight)
    elev = max(0, elevations[self.location])
    return self.sight + elev * elevation_mult


def _facing_toward(grid: HexGrid, from_idx: int, to_idx: int) -> int:
    """Direction index (0–5) from from_idx toward to_idx."""
    rel = grid.index_to_hexposition(to_idx, from_idx)
    # Find closest cardinal direction
    dirs = HexPosition.directions()
    best_dir = 0
    best_dot = -999
    for i, d in enumerate(dirs):
        # Dot product in cube coords
        dot = rel.q * d.q + rel.r * d.r + rel.s * d.s
        if dot > best_dot:
            best_dot = dot
            best_dir = i
    return best_dir


def _facing_away(grid: HexGrid, from_idx: int, to_idx: int) -> int:
    """Direction index (0–5) pointing away from to_idx."""
    return (_facing_toward(grid, from_idx, to_idx) + 3) % 6


def surround(pieces: list[Piece], target: int,
             grid: HexGrid, elevations: np.ndarray,
             mode: SurroundMode = SurroundMode.GUARD,
             max_ring: int = 3,
             countries: np.ndarray = None,
             elevation_mult: float = 0.005
             ) -> dict[str, InstructionList]:
    """Assign pieces to surround positions using Hungarian matching.
    
    Args:
        pieces: Pieces to assign (must have locations)
        target: Hex index to surround
        grid: HexGrid
        elevations: Elevation array
        mode: FEED (face inward) or GUARD (face outward)
        max_ring: Maximum ring distance from target
        countries: Country array (optional, for impassable)
        elevation_mult: Sight bonus per elevation unit
    
    Returns:
        Dict mapping piece.id → InstructionList with path + final rotation
    """
    # 1. Generate candidate positions (rings 1..max_ring)
    candidates = []
    for ring in range(1, max_ring + 1):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, target)
            if idx < 0 or idx in grid.invalidRegion:
                continue
            if elevations[idx] <= 0:
                continue
            if countries is not None and countries[idx] < 0:
                continue
            candidates.append((idx, ring))
    
    if not candidates or not pieces:
        return {}
    
    n_pieces = len(pieces)
    n_positions = len(candidates)
    
    # 2. Build cost matrix: pieces × positions
    INF = 1e9
    cost = np.full((n_pieces, n_positions), INF)
    
    for i, piece in enumerate(pieces):
        if piece.location is None or piece.location < 0:
            continue
        
        # Effective sight at current location (conservative — use current elev)
        eff_sight = piece.effective_sight(elevations, elevation_mult)
        
        for j, (pos_idx, ring_dist) in enumerate(candidates):
            # Can piece see target from this position?
            if ring_dist > eff_sight:
                continue  # can't see target from here
            
            # Pathfind cost
            path = piece.pathfind(pos_idx, grid, elevations, countries)
            if not path:
                continue  # unreachable
            
            # Cost = path length (weighted by movement costs)
            path_cost = 0.0
            for k in range(len(path) - 1):
                path_cost += _move_cost(elevations, path[k], path[k + 1])
            
            cost[i, j] = path_cost
    
    # 3. Hungarian matching
    row_ind, col_ind = linear_sum_assignment(cost)
    
    # 4. Generate instruction lists
    assignments = {}
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue  # unmatched
        
        piece = pieces[i]
        pos_idx = candidates[j][0]
        
        path = piece.pathfind(pos_idx, grid, elevations, countries)
        if len(path) < 2:
            continue
        
        rules = InstructionList.path_to_rules(path, grid, start_facing=piece.facing)
        
        # Append final facing rotation
        if mode == SurroundMode.FEED:
            desired_facing = _facing_toward(grid, pos_idx, target)
        else:
            desired_facing = _facing_away(grid, pos_idx, target)
        
        # Calculate rotations needed to reach desired facing
        # Current facing after path (last rule's facing)
        # Simulate to get final facing — or just compute from path end
        path_end_facing = _facing_toward(grid, path[-2], path[-1]) if len(path) >= 2 else piece.facing
        
        # Shortest rotation to desired
        diff = (desired_facing - path_end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))
        
        assignments[piece.id] = InstructionList(rules, cursor=0, patrol=False)
    
    return assignments

Can you build a demo using a gameboard that shows a bunch of peices moving to feed the queen?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
# --- Build the board ---
board = PieceBoard.hilly(rings=7, radius=20, seed=42)
grid = board.grid
elevs = board.elevations

# Place the queen near center on high ground
mid = grid.middle
queen = board.add_piece(mid, PieceType.QUEEN, facing=0, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
queen.food = 2.0  # hungry queen

# Scatter pawns and a knight around — various distances
offsets = [
    (-3, -2), (-4, 1), (2, -3), (3, 2), (5, 0), (-2, 4), (1, -5), (-5, -1)
]
feeders = []
for i, (dq, dr) in enumerate(offsets):
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, mid)
    if idx < 0 or elevs[idx] <= 0:
        continue
    ptype = PieceType.KNIGHT if i == 0 else PieceType.PAWN
    p = board.add_piece(idx, ptype, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList())
    p.food = 8.0  # well-fed, ready to share
    feeders.append(p)

print(f"Queen at hex {mid}, {len(feeders)} feeders placed")

# --- Run surround with FEED mode ---
orders = surround(feeders, queen.location, grid, elevs,
                  mode=SurroundMode.FEED, max_ring=2)

print(f"Assigned {len(orders)} pieces to surround positions")
for p in feeders:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name} ({p.piece_type.name}): {len(p.instructions.rules)} instructions")

# --- Render ---
board.render(show_plan=True, num_turns=8)
show(board)
```

This sets up:
- **Queen** in the center with low food and a PAUSE loop (she stays put)
- **8 feeders** scattered 3–5 hexes away, all well-fed
- `surround(mode=FEED)` assigns each to a ring-1 or ring-2 position, facing inward
- Plan overlay shows the dashed-arrow paths converging on the queen

If some feeders don't get assigned (too far, blocked by water, etc.), you'll see fewer than 8 in the printout — that's the Hungarian matching doing its job, only assigning feasible positions.

In [ ]:
# --- Build the board ---
board = PieceBoard.hilly(rings=7, radius=20, seed=42)
grid = board.grid
elevs = board.elevations

# Place the queen near center on high ground
mid = grid.middle
queen = board.add_piece(mid, PieceType.QUEEN, facing=0, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
queen.food = 2.0  # hungry queen

# Scatter pawns and a knight around — various distances
offsets = [
    (-3, -2), (-4, 1), (2, -3), (3, 2), (5, 0), (-2, 4), (1, -5), (-5, -1)
]
feeders = []
for i, (dq, dr) in enumerate(offsets):
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, mid)
    if idx < 0 or elevs[idx] <= 0:
        continue
    ptype = PieceType.KNIGHT if i == 0 else PieceType.PAWN
    p = board.add_piece(idx, ptype, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 8.0  # well-fed, ready to share
    feeders.append(p)

print(f"Queen at hex {mid}, {len(feeders)} feeders placed")

# --- Run surround with FEED mode ---
orders = surround(feeders, queen.location, grid, elevs,
                  mode=SurroundMode.FEED, max_ring=2)

print(f"Assigned {len(orders)} pieces to surround positions")
for p in feeders:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name} ({p.piece_type.name}): {len(p.instructions.rules)} instructions")

# --- Render ---
#board.render(show_plan=True, num_turns=8)
#show(board)


Fix?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



`InstructionList` requires a `rules` argument. Change the empty ones to `InstructionList([])`:

```python
# --- Build the board ---
board = PieceBoard.hilly(rings=7, radius=20, seed=42)
grid = board.grid
elevs = board.elevations

# Place the queen near center on high ground
mid = grid.middle
queen = board.add_piece(mid, PieceType.QUEEN, facing=0, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
queen.food = 2.0  # hungry queen

# Scatter pawns and a knight around — various distances
offsets = [
    (-3, -2), (-4, 1), (2, -3), (3, 2), (5, 0), (-2, 4), (1, -5), (-5, -1)
]
feeders = []
for i, (dq, dr) in enumerate(offsets):
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, mid)
    if idx < 0 or elevs[idx] <= 0:
        continue
    ptype = PieceType.KNIGHT if i == 0 else PieceType.PAWN
    p = board.add_piece(idx, ptype, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 8.0  # well-fed, ready to share
    feeders.append(p)

print(f"Queen at hex {mid}, {len(feeders)} feeders placed")

# --- Run surround with FEED mode ---
orders = surround(feeders, queen.location, grid, elevs,
                  mode=SurroundMode.FEED, max_ring=2)

print(f"Assigned {len(orders)} pieces to surround positions")
for p in feeders:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name} ({p.piece_type.name}): {len(p.instructions.rules)} instructions")

# --- Render ---
board.render(show_plan=True, num_turns=8)
show(board)
```

How about with a pieces overlay and the queen not sitting in water?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Two issues:

1. **Queen on water** — `PieceBoard.hilly` carves a river along `r=0`, which includes the middle hex. Need to find a nearby land hex.
2. **Empty plan layer** — `show(board)` calls `__ft__()` which renders without `show_plan`. Use `PieceBoardPlan` instead.

```python
# --- Build the board ---
board = PieceBoard.hilly(rings=7, radius=20, seed=42)
grid = board.grid
elevs = board.elevations

# Find nearest land hex to center for queen
mid = grid.middle
queen_idx = None
for dist in range(0, 5):
    for hp in HexPosition.origin().ring(dist):
        idx = grid.hexposition_to_index(hp, mid)
        if idx >= 0 and elevs[idx] > 0:
            queen_idx = idx
            break
    if queen_idx is not None:
        break

print(f"Queen placed at hex {queen_idx} (elev={elevs[queen_idx]:.0f})")

queen = board.add_piece(queen_idx, PieceType.QUEEN, facing=0, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
queen.food = 2.0

# Scatter feeders — skip any that land on water
offsets = [
    (-3, -2), (-4, 1), (2, -3), (3, 2), (5, 0), (-2, 4), (1, -5), (-5, -1)
]
feeders = []
for i, (dq, dr) in enumerate(offsets):
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, queen_idx)
    if idx < 0 or elevs[idx] <= 0:
        continue
    ptype = PieceType.KNIGHT if i == 0 else PieceType.PAWN
    p = board.add_piece(idx, ptype, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 8.0
    feeders.append(p)

print(f"{len(feeders)} feeders placed")

# --- Surround with FEED mode ---
orders = surround(feeders, queen.location, grid, elevs,
                  mode=SurroundMode.FEED, max_ring=2)

print(f"Assigned {len(orders)} pieces")
for p in feeders:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name} ({p.piece_type.name}): {len(p.instructions.rules)} instructions")

# --- Render with plan overlay ---
PieceBoardPlan(board, num_turns=8)
```

The key changes:
- Search outward from center until we find a land hex for the queen
- Use `PieceBoardPlan(board)` instead of `show(board)` to get the dashed-arrow movement plans rendered

In [ ]:
# --- Build the board ---
board = PieceBoard.hilly(rings=7, radius=20, seed=42)
grid = board.grid
elevs = board.elevations

# Find nearest land hex to center for queen
mid = grid.middle
queen_idx = None
for dist in range(0, 5):
    for hp in HexPosition.origin().ring(dist):
        idx = grid.hexposition_to_index(hp, mid)
        if idx >= 0 and elevs[idx] > 0:
            queen_idx = idx
            break
    if queen_idx is not None:
        break

print(f"Queen placed at hex {queen_idx} (elev={elevs[queen_idx]:.0f})")

queen = board.add_piece(queen_idx, PieceType.QUEEN, facing=0, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
queen.food = 2.0

# Scatter feeders — skip any that land on water
offsets = [
    (-3, -2), (-4, 1), (2, -3), (3, 2), (5, 0), (-2, 4), (1, -5), (-5, -1)
]
feeders = []
for i, (dq, dr) in enumerate(offsets):
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, queen_idx)
    if idx < 0 or elevs[idx] <= 0:
        continue
    ptype = PieceType.KNIGHT if i == 0 else PieceType.PAWN
    p = board.add_piece(idx, ptype, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 8.0
    feeders.append(p)

print(f"{len(feeders)} feeders placed")

# --- Surround with FEED mode ---
orders = surround(feeders, queen.location, grid, elevs,
                  mode=SurroundMode.FEED, max_ring=2)

print(f"Assigned {len(orders)} pieces")
for p in feeders:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name} ({p.piece_type.name}): {len(p.instructions.rules)} instructions")

# --- Render with plan overlay ---
#PieceBoardPlan(board, num_turns=8)


this is so perfect. Maybe we could add a 
```
@patch
def facing_bar(self: DiagramGlyphs, cx, cy, facing: int,
               length=None, offset=None, stroke=None,
               stroke_width=None, opacity=None) -> str:
    """Short flat bar on the 'front' side of the piece — football blocking style."""
    import math
    length = length or self.size * 0.7
    offset = offset or self.size * 0.55
    stroke = stroke or self.color
    stroke_width = stroke_width or max(2, self.size * 0.15)
    opacity = opacity if opacity is not None else self.opacity

    # Facing angle: direction 0 = SE in your hex layout
    # Adjust this base angle to match your HexPosition.directions() order
    angle = math.radians(60 * facing)  # tune to your coord system

    # Center of the bar, pushed outward along facing
    bx = cx + offset * math.cos(angle)
    by = cy + offset * math.sin(angle)




    # Perpendicular endpoints
    perp = angle + math.pi / 2
    half = length / 2
    x1, y1 = bx + half * math.cos(perp), by + half * math.sin(perp)
    x2, y2 = bx - half * math.cos(perp), by - half * math.sin(perp)

    return (f'<line x1="{x1:.1f}" y1="{y1:.1f}" '
            f'x2="{x2:.1f}" y2="{y2:.1f}" '
            f'stroke="{stroke}" stroke-width="{stroke_width:.1f}" '
            f'stroke-linecap="round" opacity="{opacity}"/>\n')
            ```
            at the end to see where the pieces are facing

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The `facing_bar` is already on `DiagramGlyphs` — we just need to call it at each piece's final step position. Cleanest approach is a small helper patched onto `PieceBoard`:

```python
@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue
        
        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center
        color = piece.flag.primary if piece.flag else "#333"
        g = DiagramGlyphs(color=color, size=self.grid.radius * 0.45)
        svg += g.facing_bar(c.x, c.y, last.facing)
    
    return svg


@patch
def render_with_facing(self: PieceBoard, num_turns: int = 8) -> str:
    """Board render with plan overlay + final facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("pieces",  self._pieces_overlay())
    self.grid.builder.adjust("plan",    self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing",  self._facing_overlay(num_turns))
    
    xs = [h.center.x for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    ys = [h.center.y for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()
```

Then render with:

```python
show(NotStr(board.render_with_facing(num_turns=8)))
```

The facing bar appears at each piece's predicted final hex, perpendicular to their facing direction — so you can see all the pawns pointing inward toward the queen. The knight's longer path will show it rotating at the end to face her too.

In [ ]:
@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue
        
        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center
        color = piece.flag.primary if piece.flag else "#333"
        g = DiagramGlyphs(color=color, size=self.grid.radius * 0.45)
        svg += g.facing_bar(c.x, c.y, last.facing)
    
    return svg


@patch
def render_with_facing(self: PieceBoard, num_turns: int = 8) -> str:
    """Board render with plan overlay + final facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("pieces",  self._pieces_overlay())
    self.grid.builder.adjust("plan",    self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing",  self._facing_overlay(num_turns))
    
    xs = [h.center.x for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    ys = [h.center.y for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()


In [ ]:
@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue

        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center
        color = piece.flag.primary if piece.flag else "#fff"
        
        # Dark outline pass first, then bright bar on top
        g_outline = DiagramGlyphs(color="#222222",
                                  size=self.grid.radius * 0.55,
                                  stroke_width=4.5, opacity=0.9)
        g_bar = DiagramGlyphs(color="#FFFFFF",
                              size=self.grid.radius * 0.55,
                              stroke_width=2.5, opacity=1.0)
        
        svg += g_outline.facing_bar(c.x, c.y, last.facing)
        svg += g_bar.facing_bar(c.x, c.y, last.facing)

    return svg


In [ ]:
@patch
def render_with_facing(self: PieceBoard, num_turns: int = 8) -> str:
    """Board render with plan overlay + final facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("pieces",  self._pieces_overlay())
    self.grid.builder.adjust("plan",    self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing",  self._facing_overlay(num_turns))
    
    xs = [h.center.x for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    ys = [h.center.y for i, h in enumerate(self.grid.hexes) if i not in self.grid.invalidRegion]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()



In [ ]:
#show(NotStr(board.render_with_facing(num_turns=8)))

I can't really see the bars. Are they under? should we use a different color?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The bars are there but tiny (6px) and `#dee5a5` — same color as the dashed arrows, so they vanish. Two fixes: make them thicker and use a high-contrast color with a dark outline:

```python
@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue

        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center
        color = piece.flag.primary if piece.flag else "#fff"
        
        # Dark outline pass first, then bright bar on top
        g_outline = DiagramGlyphs(color="#222222",
                                  size=self.grid.radius * 0.55,
                                  stroke_width=4.5, opacity=0.9)
        g_bar = DiagramGlyphs(color="#FFFFFF",
                              size=self.grid.radius * 0.55,
                              stroke_width=2.5, opacity=1.0)
        
        svg += g_outline.facing_bar(c.x, c.y, last.facing)
        svg += g_bar.facing_bar(c.x, c.y, last.facing)

    return svg
```

White on dark outline gives maximum contrast against any terrain color. If you want to keep per-kingdom colors, swap `#FFFFFF` back to `color` — but white tends to pop the most against the parchment/earth tones.

so they should extend further towards the edge of the hex we also want thing rotated to face the queen

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Two issues to fix:

1. **Angle formula is off by 120°** — `60 * facing` assumes facing=0 is East, but your direction order is SW=0, W=1, NW=2, NE=3, E=4, SE=5, so it needs `60 * facing + 120`
2. **Bar too small** — push `offset` and `length` out toward the hex edge

```python
@patch
def facing_bar(self: DiagramGlyphs, cx, cy, facing: int,
               length=None, offset=None, stroke=None,
               stroke_width=None, opacity=None) -> str:
    """Short flat bar on the 'front' side of the piece — football blocking style."""
    import math
    length       = length       or self.size * 1.2          # wider bar
    offset       = offset       or self.size * 1.1          # pushed toward hex edge
    stroke       = stroke       or self.color
    stroke_width = stroke_width or max(2, self.size * 0.15)
    opacity      = opacity if opacity is not None else self.opacity

    # SW=0,W=1,NW=2,NE=3,E=4,SE=5 → pixel angles 120,180,240,300,0,60
    angle = math.radians(60 * facing + 120)

    bx = cx + offset * math.cos(angle)
    by = cy + offset * math.sin(angle)

    perp = angle + math.pi / 2
    half = length / 2
    x1, y1 = bx + half * math.cos(perp), by + half * math.sin(perp)
    x2, y2 = bx - half * math.cos(perp), by - half * math.sin(perp)

    return (f'<line x1="{x1:.1f}" y1="{y1:.1f}" '
            f'x2="{x2:.1f}" y2="{y2:.1f}" '
            f'stroke="{stroke}" stroke-width="{stroke_width:.1f}" '
            f'stroke-linecap="round" opacity="{opacity}"/>\n')


@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    r = self.grid.radius
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue

        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center

        # Size drives offset/length via the multipliers above
        g_outline = DiagramGlyphs(color="#111111", size=r * 0.7,
                                  stroke_width=5.0, opacity=0.85)
        g_bar     = DiagramGlyphs(color="#FFFFFF",  size=r * 0.7,
                                  stroke_width=2.8, opacity=1.0)

        svg += g_outline.facing_bar(c.x, c.y, last.facing)
        svg += g_bar.facing_bar(c.x, c.y, last.facing)

    return svg
```

Then re-render:

```python
show(NotStr(board.render_with_facing(num_turns=8)))
```

The bars should now sit near the hex edge and point toward the queen from each feeder's final position.

In [ ]:
@patch
def facing_bar(self: DiagramGlyphs, cx, cy, facing: int,
               length=None, offset=None, stroke=None,
               stroke_width=None, opacity=None) -> str:
    """Short flat bar on the 'front' side of the piece — football blocking style."""
    import math
    length       = length       or self.size * 1.2          # wider bar
    offset       = offset       or self.size * 1.1          # pushed toward hex edge
    stroke       = stroke       or self.color
    stroke_width = stroke_width or max(2, self.size * 0.15)
    opacity      = opacity if opacity is not None else self.opacity

    # SW=0,W=1,NW=2,NE=3,E=4,SE=5 → pixel angles 120,180,240,300,0,60
    angle = math.radians(60 * facing + 120)

    bx = cx + offset * math.cos(angle)
    by = cy + offset * math.sin(angle)

    perp = angle + math.pi / 2
    half = length / 2
    x1, y1 = bx + half * math.cos(perp), by + half * math.sin(perp)
    x2, y2 = bx - half * math.cos(perp), by - half * math.sin(perp)

    return (f'<line x1="{x1:.1f}" y1="{y1:.1f}" '
            f'x2="{x2:.1f}" y2="{y2:.1f}" '
            f'stroke="{stroke}" stroke-width="{stroke_width:.1f}" '
            f'stroke-linecap="round" opacity="{opacity}"/>\n')


@patch
def _facing_overlay(self: PieceBoard, num_turns: int = 8) -> str:
    """Facing bars at each piece's final simulated position."""
    svg = ""
    r = self.grid.radius
    for piece in self.pieces:
        if not piece.instructions.rules:
            continue
        steps = piece.simulate(self.grid, self.elevations,
                               num_turns=num_turns,
                               countries=self.countries)
        if not steps:
            continue

        last = steps[-1]
        c = self.grid.hexes[last.hex_idx].center

        # Size drives offset/length via the multipliers above
        g_outline = DiagramGlyphs(color="#111111", size=r * 0.7,
                                  stroke_width=5.0, opacity=0.85)
        g_bar     = DiagramGlyphs(color="#FFFFFF",  size=r * 0.7,
                                  stroke_width=2.8, opacity=1.0)

        svg += g_outline.facing_bar(c.x, c.y, last.facing)
        svg += g_bar.facing_bar(c.x, c.y, last.facing)

    return svg


In [ ]:
#show(NotStr(board.render_with_facing(num_turns=8)))

So I think we are ready for our sightOverlay. we want to see which hexes are covered by a list of pieces. perhaps the overlay could be a dotted one similar to food, but with the kingdom color

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@patch
def hexes_in_sight(self: Piece, grid: HexGrid, elevations: np.ndarray,
                   elevation_mult: float = 0.005,
                   facing_only: bool = False,
                   allowed_rotations: int = 1) -> list[int]:
    """All hex indices visible from this piece's location."""
    if self.location is None or self.location < 0:
        return []
    
    elev = max(0, elevations[self.location])
    effective_sight = self.sight + elev * elevation_mult
    facing_dir = HexPosition.directions()[self.facing % 6]
    
    visible = []
    for idx in grid.indices_in_range(self.location, int(effective_sight) + 1):
        relative = grid.index_to_hexposition(idx, self.location)
        if abs(relative) > effective_sight:
            continue
        if facing_only and not hex_in_cone(relative, facing_dir, allowed_rotations):
            continue
        visible.append(idx)
    return visible


@patch
def sight_overlay(self: PieceBoard, pieces: list[Piece],
                  elevation_mult: float = 0.005,
                  opacity: float = 0.22,
                  facing_only: bool = False,
                  allowed_rotations: int = 1) -> str:
    """Dotted sight-range overlay, one color per kingdom."""
    svg = ""
    r = self.grid.radius
    dot_r   = max(1.5, r * 0.10)
    spacing = max(5.0, r * 0.32)

    # Group by kingdom color
    color_to_hexes: dict[str, set[int]] = {}
    for piece in pieces:
        color = piece.flag.primary if piece.flag else "#888"
        visible = piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=elevation_mult,
            facing_only=facing_only,
            allowed_rotations=allowed_rotations
        )
        color_to_hexes.setdefault(color, set()).update(visible)

    for color, hex_indices in color_to_hexes.items():
        pat_id = f"sight_{color.replace('#','')}"
        # Register dot pattern as a def
        pat_svg = (
            f'<pattern id="{pat_id}" x="0" y="0" '
            f'width="{spacing:.1f}" height="{spacing:.1f}" '
            f'patternUnits="userSpaceOnUse">'
            f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
            f'r="{dot_r:.1f}" fill="{color}"/>'
            f'</pattern>'
        )
        self.grid.builder.add_definition(
            SVGDef("", pat_id, pat_svg, raw=True)
        )

        for idx in sorted(hex_indices):
            if idx < 0 or idx >= len(self.grid.hexes):
                continue
            h = self.grid.hexes[idx]
            pts = " ".join(f"{v.x},{v.y}" for v in h.v)
            svg += (
                f'<polygon points="{pts}" '
                f'fill="url(#{pat_id})" opacity="{opacity:.2f}" '
                f'stroke="{color}" stroke-width="0.6" '
                f'stroke-dasharray="3,2" stroke-opacity="0.4"/>\n'
            )
    return svg


@patch
def render_with_sight(self: PieceBoard, pieces: list[Piece],
                      num_turns: int = 8,
                      facing_only: bool = False) -> str:
    """Board with sight overlay + movement plans + facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("sight",  self.sight_overlay(pieces, facing_only=facing_only))
    self.grid.builder.adjust("pieces", self._pieces_overlay())
    self.grid.builder.adjust("plan",   self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing", self._facing_overlay(num_turns))

    xs = [h.center.x for h in self.grid.hexes]
    ys = [h.center.y for h in self.grid.hexes]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()
```

Then render with:

```python
# All pieces — see the full combined coverage
show(NotStr(board.render_with_sight(feeders + [queen], num_turns=8)))

# Or just feeders at their *final* positions — pass their post-plan locations
show(NotStr(board.render_with_sight(feeders, num_turns=8)))
```

The sight layer sits between terrain and pieces so the dots are visible but don't obscure the movement arrows or piece icons. Hexes covered by multiple pieces of the same kingdom just get the same pattern — no double-counting.

In [ ]:
@patch
def hexes_in_sight(self: Piece, grid: HexGrid, elevations: np.ndarray,
                   elevation_mult: float = 0.005,
                   facing_only: bool = False) -> list[int]:
    """All hex indices visible from this piece's location."""
    if self.location is None or self.location < 0:
        return []
    
    elev = max(0, elevations[self.location])
    effective_sight = int(self.sight + elev * elevation_mult)
    
    if facing_only:
        facing_dir = HexPosition.directions()[self.facing % 6]
        hex_positions = field_of_view(HexPosition.origin(), facing_dir, effective_sight)
        return [
            idx for hp in hex_positions
            if (idx := grid.hexposition_to_index(hp, self.location)) >= 0
        ]
    else:
        return grid.indices_in_range(self.location, effective_sight)


In [ ]:



@patch
def sight_overlay(self: PieceBoard, pieces: list[Piece],
                  elevation_mult: float = 0.005,
                  opacity: float = 0.22,
                  facing_only: bool = False,
                  allowed_rotations: int = 1) -> str:
    """Dotted sight-range overlay, one color per kingdom."""
    svg = ""
    r = self.grid.radius
    dot_r   = max(1.5, r * 0.10)
    spacing = max(5.0, r * 0.32)

    # Group by kingdom color
    color_to_hexes: dict[str, set[int]] = {}
    for piece in pieces:
        color = piece.flag.primary if piece.flag else "#888"
        visible = piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=elevation_mult,
            facing_only=facing_only,
            allowed_rotations=allowed_rotations
        )
        color_to_hexes.setdefault(color, set()).update(visible)

    for color, hex_indices in color_to_hexes.items():
        pat_id = f"sight_{color.replace('#','')}"
        # Register dot pattern as a def
        pat_svg = (
            f'<pattern id="{pat_id}" x="0" y="0" '
            f'width="{spacing:.1f}" height="{spacing:.1f}" '
            f'patternUnits="userSpaceOnUse">'
            f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
            f'r="{dot_r:.1f}" fill="{color}"/>'
            f'</pattern>'
        )
        self.grid.builder.add_definition(
            SVGDef("", pat_id, pat_svg, raw=True)
        )

        for idx in sorted(hex_indices):
            if idx < 0 or idx >= len(self.grid.hexes):
                continue
            h = self.grid.hexes[idx]
            pts = " ".join(f"{v.x},{v.y}" for v in h.v)
            svg += (
                f'<polygon points="{pts}" '
                f'fill="url(#{pat_id})" opacity="{opacity:.2f}" '
                f'stroke="{color}" stroke-width="0.6" '
                f'stroke-dasharray="3,2" stroke-opacity="0.4"/>\n'
            )
    return svg


@patch
def render_with_sight(self: PieceBoard, pieces: list[Piece],
                      num_turns: int = 8,
                      facing_only: bool = False) -> str:
    """Board with sight overlay + movement plans + facing bars."""
    self._apply_terrain()
    self.grid.update()
    self.grid.builder.adjust("sight",  self.sight_overlay(pieces, facing_only=facing_only))
    self.grid.builder.adjust("pieces", self._pieces_overlay())
    self.grid.builder.adjust("plan",   self._plan_overlay(num_turns))
    self.grid.builder.adjust("facing", self._facing_overlay(num_turns))

    xs = [h.center.x for h in self.grid.hexes]
    ys = [h.center.y for h in self.grid.hexes]
    pad = self.grid.radius * 1.5
    self.grid.builder.width  = int(max(xs) + pad)
    self.grid.builder.height = int(max(ys) + pad)
    return self.grid.builder.xml()


 think we need to work on field of view which seems to go wider that 120. in ring 1 it is fine. but in ring 2 it should go straing out from its edges but it bends

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The bug is in `hex_in_cone` — it checks the direction of the **first step** of the path to the hex, which is correct at ring 1 but at ring 2+ a hex "around the corner" can have its first step in an allowed direction even though the hex is geometrically outside the 120° wedge.

For example, facing NE with hex at NW+W=(-1,-1,2): the straight line from origin passes through NW first (which is allowed), so it's incorrectly included. But it's clearly outside the 120° cone.

The fix: use an exact **half-plane test** in cube coordinates. For a 120° cone facing direction D=(dq,dr,ds), hex P is in the cone iff each nonzero component of D has the same sign in P:

```python
def field_of_view(origin: HexPosition, facing: HexPosition, max_distance: int) -> list[HexPosition]:
    """Get all hexes within a 120° field of view (±60° from facing) up to max_distance.
    
    Uses exact half-plane test — cone stays straight at all ranges.
    For facing NE=(1,-1,0): requires pq≥0 AND pr≤0 at any distance.
    """
    fq, fr, fs = facing.q, facing.r, facing.s

    def in_cone(rel: HexPosition) -> bool:
        pq, pr, ps = rel.q, rel.r, rel.s
        if fq > 0 and pq < 0: return False
        if fq < 0 and pq > 0: return False
        if fr > 0 and pr < 0: return False
        if fr < 0 and pr > 0: return False
        if fs > 0 and ps < 0: return False
        if fs < 0 and ps > 0: return False
        return True

    results = []
    for radius in range(1, max_distance + 1):
        for hex_pos in origin.ring(radius):
            if in_cone(hex_pos - origin):
                results.append(hex_pos)
    return results
```

The pattern is elegant — each of the 6 facing directions has exactly 2 nonzero cube components, and those two sign conditions define the straight-edged wedge. You can verify:

| Facing | Condition |
|--------|-----------|
| NE (1,-1,0) | q≥0 AND r≤0 |
| E  (1,0,-1) | q≥0 AND s≤0 |
| SE (0,1,-1) | r≥0 AND s≤0 |
| SW (-1,1,0) | q≤0 AND r≥0 |
| W  (-1,0,1) | q≤0 AND s≥0 |
| NW (0,-1,1) | r≤0 AND s≥0 |

No other changes needed — `hexes_in_sight` already uses `field_of_view` so the fix propagates automatically.

These wound up too suttble. I wonder if darkPrimary or baseComp or perhaps larger dots makes more sense.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Looking at `CountryFlag`, `darkPrimary` is the richest option — it's the full-saturation dark version of the kingdom color, so it'll read clearly against the parchment/green terrain without being garish. `baseComp` is complementary which could clash badly.

Two knobs to turn — color and dot size:

```python
@patch
def sight_overlay(self: PieceBoard, pieces: list[Piece],
                  elevation_mult: float = 0.005,
                  opacity: float = 0.35,          # up from 0.22
                  facing_only: bool = False,
                  allowed_rotations: int = 1,
                  color_attr: str = "darkPrimary"  # "primary" | "darkPrimary" | "baseComp"
                  ) -> str:
    """Dotted sight-range overlay, one color per kingdom."""
    svg = ""
    r = self.grid.radius
    dot_r   = max(2.5, r * 0.18)   # up from 0.10
    spacing = max(5.0, r * 0.32)

    color_to_hexes: dict[str, set[int]] = {}
    for piece in pieces:
        color = getattr(piece.flag, color_attr, "#888") if piece.flag else "#888"
        visible = piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=elevation_mult,
            facing_only=facing_only,
            allowed_rotations=allowed_rotations
        )
        color_to_hexes.setdefault(color, set()).update(visible)

    for color, hex_indices in color_to_hexes.items():
        pat_id = f"sight_{color.replace('#','')}"
        pat_svg = (
            f'<pattern id="{pat_id}" x="0" y="0" '
            f'width="{spacing:.1f}" height="{spacing:.1f}" '
            f'patternUnits="userSpaceOnUse">'
            f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
            f'r="{dot_r:.1f}" fill="{color}"/>'
            f'</pattern>'
        )
        self.grid.builder.add_definition(
            SVGDef("", pat_id, pat_svg, raw=True)
        )

        for idx in sorted(hex_indices):
            if idx < 0 or idx >= len(self.grid.hexes):
                continue
            h = self.grid.hexes[idx]
            pts = " ".join(f"{v.x},{v.y}" for v in h.v)
            svg += (
                f'<polygon points="{pts}" '
                f'fill="url(#{pat_id})" opacity="{opacity:.2f}" '
                f'stroke="{color}" stroke-width="0.8" '
                f'stroke-dasharray="3,2" stroke-opacity="0.5"/>\n'
            )
    return svg
```

Then test the three options:

```python
# darkPrimary — recommended
show(NotStr(board.render_with_sight(feeders + [queen], num_turns=8)))

# or try baseComp for maximum contrast (complementary color)
show(NotStr(board.render_with_sight(feeders + [queen], num_turns=8)))
# (change color_attr="baseComp" in sight_overlay call inside render_with_sight)
```

My bet is `darkPrimary` at `opacity=0.35`, `dot_r = r * 0.18` is the sweet spot — visible but not overwhelming.

In [ ]:
@patch
def sight_overlay(self: PieceBoard, pieces: list[Piece],
                  elevation_mult: float = 0.005,
                  opacity: float = 0.35,          # up from 0.22
                  facing_only: bool = False,
                  allowed_rotations: int = 1,
                  color_attr: str = "darkPrimary"  # "primary" | "darkPrimary" | "baseComp"
                  ) -> str:
    """Dotted sight-range overlay, one color per kingdom."""
    svg = ""
    r = self.grid.radius
    dot_r   = max(2.5, r * 0.18)   # up from 0.10
    spacing = max(5.0, r * 0.32)

    color_to_hexes: dict[str, set[int]] = {}
    for piece in pieces:
        color = getattr(piece.flag, color_attr, "#888") if piece.flag else "#888"
        visible = piece.hexes_in_sight(
            self.grid, self.elevations,
            elevation_mult=elevation_mult,
            facing_only=facing_only
        )
        color_to_hexes.setdefault(color, set()).update(visible)

    for color, hex_indices in color_to_hexes.items():
        pat_id = f"sight_{color.replace('#','')}"
        pat_svg = (
            f'<pattern id="{pat_id}" x="0" y="0" '
            f'width="{spacing:.1f}" height="{spacing:.1f}" '
            f'patternUnits="userSpaceOnUse">'
            f'<circle cx="{spacing/2:.1f}" cy="{spacing/2:.1f}" '
            f'r="{dot_r:.1f}" fill="{color}"/>'
            f'</pattern>'
        )
        self.grid.builder.add_definition(
            SVGDef("", pat_id, pat_svg, raw=True)
        )

        for idx in sorted(hex_indices):
            if idx < 0 or idx >= len(self.grid.hexes):
                continue
            h = self.grid.hexes[idx]
            pts = " ".join(f"{v.x},{v.y}" for v in h.v)
            svg += (
                f'<polygon points="{pts}" '
                f'fill="url(#{pat_id})" opacity="{opacity:.2f}" '
                f'stroke="{color}" stroke-width="0.8" '
                f'stroke-dasharray="3,2" stroke-opacity="0.5"/>\n'
            )
    return svg


Lets do a demo with one piece so I can see the cone

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

> ⚠️ Note: The dialog is large — some earlier messages have been trimmed from context. Let me know if anything seems off.

```python
# --- Fresh board ---
board = PieceBoard.hilly(rings=7, radius=20, seed=42)
grid  = board.grid
elevs = board.elevations

# Find a nice land hex near center
mid = grid.middle
scout_idx = None
for dist in range(0, 6):
    for hp in HexPosition.origin().ring(dist):
        idx = grid.hexposition_to_index(hp, mid)
        if idx >= 0 and elevs[idx] > 0:
            scout_idx = idx
            break
    if scout_idx is not None:
        break

# Single scout, facing direction 3 (NE) so cone points up-right
scout = board.add_piece(scout_idx, PieceType.KNIGHT, facing=3, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
scout.sight = 4   # decent range

print(f"Scout at hex {scout_idx} (elev={elevs[scout_idx]:.0f}), facing=3 (NE)")
print(f"Effective sight: {scout.effective_sight(elevs):.2f}")

# Cone only — shows the 120° wedge
show(NotStr(board.render_with_sight(
    [scout],
    num_turns=1,
    facing_only=True   # ← cone mode
)))
```

This will show:
- The **dotted dark cone** fanning out NE from the scout
- The **facing bar** confirming the direction
- No movement arrows since it's just PAUSEing

Try swapping `facing=0` through `5` to verify all six directions look right, then we can confirm the angle formula is calibrated.

# darkPrimary — recommended
show(NotStr(board.render_with_sight(feeders + [queen], num_turns=8)))

In [ ]:
def field_of_view(origin: HexPosition, facing: HexPosition, max_distance: int) -> list[HexPosition]:
    """Get all hexes within a 120° field of view (±60° from facing) up to max_distance.
    
    Walks the cone arc per ring — no per-hex tests needed.
    """
    left_edge = facing.rotate(1)     # start of arc (left side)
    step1 = facing.rotate(-1)        # walk from left toward center
    step2 = facing.rotate(-2)        # walk from center toward right
    
    results = []
    for r in range(1, max_distance + 1):
        current = origin + r * left_edge
        for _ in range(r):
            results.append(current)
            current = current + step1
        for _ in range(r):
            results.append(current)
            current = current + step2
        results.append(current)      # final right-edge hex
    return results


# --- Fresh board ---
board = PieceBoard.hilly(rings=7, radius=20, seed=42)
grid  = board.grid
elevs = board.elevations

# Find a nice land hex near center
mid = grid.middle
scout_idx = None
for dist in range(0, 6):
    for hp in HexPosition.origin().ring(dist):
        idx = grid.hexposition_to_index(hp, mid)
        if idx >= 0 and elevs[idx] > 0:
            scout_idx = idx
            break
    if scout_idx is not None:
        break

# Single scout, facing direction 3 (NE) so cone points up-right
scout = board.add_piece(scout_idx, PieceType.KNIGHT, facing=3, country_id=1,
                        instructions=InstructionList([Instruction.PAUSE.value], patrol=True))
scout.sight = 4   # decent range

print(f"Scout at hex {scout_idx} (elev={elevs[scout_idx]:.0f}), facing=3 (NE)")
print(f"Effective sight: {scout.effective_sight(elevs):.2f}")

# Cone only — shows the 120° wedge
show(NotStr(board.render_with_sight(
    [scout],
    num_turns=1,
    facing_only=True   # ← cone mode
)))


So I am wondering about another matching algorith. we want to cover a path so that as a piece walks along it they would be fed. so we need to put pieces on the best yields "outside of the path". and then use a similar bipartite graph to match them. I think these field of view algorithms will come in handy for getting covverage.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Brilliant — this is a **supply line** problem. Station harvesters on fertile tiles flanking the march route so they can GIVE into the path as the traveler passes. Here's my thinking:

**Phase 1 — Candidate positions**
For each hex on the path, look outward 1–2 rings for tiles that:
- Are NOT on the path itself
- Have `food_tier >= min_tier`
- Have line of sight back to the path (using FOV from candidate → path)

Score each candidate by `food_tier × path_hexes_covered`. A candidate on tier 6 land whose GIVE cone (facing inward) covers 3 path hexes is gold.

**Phase 2 — Minimum coverage**
This is really a **weighted set cover**: each candidate covers some subset of path hexes, and we want every path hex covered by at least one feeder's cone. Greedy set cover (pick highest coverage/cost ratio iteratively) gets us within ln(n) of optimal — plenty good for a game AI.

**Phase 3 — Hungarian matching**
Once we have the selected positions, build cost matrix `feeders × selected_positions` where cost = pathfind distance. Pieces with short sight get matched to positions close to the path. Hungarian assigns optimally.

**Phase 4 — Instructions**
Each assigned piece gets: `pathfind_to_rules(position)` + rotate to face nearest path hex + `[HARVEST, HARVEST, GIVE]` patrol loop.

```python
def supply_line(path: list[int], feeders: list[Piece],
                grid: HexGrid, elevations: np.ndarray,
                food_tiers: np.ndarray,
                min_tier: int = 2,
                max_offset: int = 2,
                countries: np.ndarray = None,
                elevation_mult: float = 0.005
                ) -> dict[str, InstructionList]:
    """Station feeders on fertile tiles along a march path.
    
    Args:
        path: Hex indices of the march route
        feeders: Available pieces to assign as suppliers
        grid: HexGrid
        elevations: Elevation array
        food_tiers: Per-hex food tier (0..n_tiers)
        min_tier: Minimum food tier for a candidate position
        max_offset: How many rings off-path to search
        countries: Country passability array
        elevation_mult: Sight bonus per elevation unit
    
    Returns:
        Dict mapping piece.id → InstructionList (path + face + harvest/give loop)
    """
    path_set = set(path)
    
    # --- Phase 1: Generate scored candidates ---
    # candidate = (hex_idx, facing_toward_path, food_tier, covered_path_hexes)
    candidates = {}  # idx -> (facing, tier, covered_set)
    
    for path_hex in path:
        for ring in range(1, max_offset + 1):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, path_hex)
                if idx < 0 or idx in grid.invalidRegion:
                    continue
                if elevations[idx] <= 0:
                    continue
                if idx in path_set:
                    continue
                if int(food_tiers[idx]) < min_tier:
                    continue
                if countries is not None and countries[idx] < 0:
                    continue
                
                # Face toward nearest path hex
                facing = _facing_toward(grid, idx, path_hex)
                facing_dir = HexPosition.directions()[facing]
                
                # What path hexes does this candidate's GIVE cone cover?
                elev = max(0, elevations[idx])
                eff_sight = 3 + elev * elevation_mult  # default sight for supply calc
                fov = set(field_of_view(HexPosition.origin(), facing_dir, int(eff_sight)))
                
                covered = set()
                for ph in path:
                    rel = grid.index_to_hexposition(ph, idx)
                    if rel in fov:
                        covered.add(ph)
                
                if not covered:
                    continue
                
                tier = int(food_tiers[idx])
                
                # Keep best coverage per candidate hex
                if idx not in candidates or len(covered) > len(candidates[idx][2]):
                    candidates[idx] = (facing, tier, covered)
    
    if not candidates:
        return {}
    
    # --- Phase 2: Greedy set cover ---
    uncovered = set(path)
    selected = []  # (idx, facing, tier)
    remaining = dict(candidates)
    
    while uncovered and remaining:
        # Pick candidate with best score: coverage × tier
        best_idx = max(remaining, key=lambda i: (
            len(remaining[i][2] & uncovered) * remaining[i][1]
        ))
        facing, tier, covered = remaining.pop(best_idx)
        
        newly_covered = covered & uncovered
        if not newly_covered:
            continue
        
        selected.append((best_idx, facing, tier))
        uncovered -= newly_covered
    
    if not selected:
        return {}
    
    # --- Phase 3: Hungarian matching ---
    n_feeders = len(feeders)
    n_positions = len(selected)
    
    INF = 1e9
    cost = np.full((n_feeders, n_positions), INF)
    
    for i, piece in enumerate(feeders):
        if piece.location is None or piece.location < 0:
            continue
        for j, (pos_idx, _, _) in enumerate(selected):
            p = piece.pathfind(pos_idx, grid, elevations, countries)
            if not p:
                continue
            c = sum(_move_cost(elevations, p[k], p[k+1]) for k in range(len(p)-1))
            cost[i, j] = c
    
    row_ind, col_ind = linear_sum_assignment(cost)
    
    # --- Phase 4: Build instructions ---
    assignments = {}
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue
        
        piece = feeders[i]
        pos_idx, facing, _ = selected[j]
        
        # Path to position
        p = piece.pathfind(pos_idx, grid, elevations, countries)
        if len(p) < 2:
            continue
        rules = InstructionList.path_to_rules(p, grid, start_facing=piece.facing)
        
        # Rotate to face path
        path_end_facing = _facing_toward(grid, p[-2], p[-1]) if len(p) >= 2 else piece.facing
        diff = (facing - path_end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))
        
        # Patrol loop: harvest then give
        rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
        
        assignments[piece.id] = InstructionList(rules, cursor=0, patrol=True)
    
    return assignments
```

Usage:

```python
# Queen's march route
march_path = queen.pathfind(target_hex, grid, elevations)

# Station pawns as supply chain
orders = supply_line(march_path, pawns, grid, elevations, food_tiers)
for p in pawns:
    if p.id in orders:
        p.instructions = orders[p.id]
```

The greedy set cover in Phase 2 is the key insight — it picks the fewest positions needed while prioritizing fertile tiles. Then Hungarian minimizes total marching distance to get everyone in place.

Want to build a visual demo of this?

In [ ]:
def supply_line(path: list[int], feeders: list[Piece],
                grid: HexGrid, elevations: np.ndarray,
                food_tiers: np.ndarray,
                min_tier: int = 2,
                max_offset: int = 2,
                countries: np.ndarray = None,
                elevation_mult: float = 0.005
                ) -> dict[str, InstructionList]:
    """Station feeders on fertile tiles along a march path.
    
    Args:
        path: Hex indices of the march route
        feeders: Available pieces to assign as suppliers
        grid: HexGrid
        elevations: Elevation array
        food_tiers: Per-hex food tier (0..n_tiers)
        min_tier: Minimum food tier for a candidate position
        max_offset: How many rings off-path to search
        countries: Country passability array
        elevation_mult: Sight bonus per elevation unit
    
    Returns:
        Dict mapping piece.id → InstructionList (path + face + harvest/give loop)
    """
    path_set = set(path)
    
    # --- Phase 1: Generate scored candidates ---
    # candidate = (hex_idx, facing_toward_path, food_tier, covered_path_hexes)
    candidates = {}  # idx -> (facing, tier, covered_set)
    
    for path_hex in path:
        for ring in range(1, max_offset + 1):
            for hp in HexPosition.origin().ring(ring):
                idx = grid.hexposition_to_index(hp, path_hex)
                if idx < 0 or idx in grid.invalidRegion:
                    continue
                if elevations[idx] <= 0:
                    continue
                if idx in path_set:
                    continue
                if int(food_tiers[idx]) < min_tier:
                    continue
                if countries is not None and countries[idx] < 0:
                    continue
                
                # Face toward nearest path hex
                facing = _facing_toward(grid, idx, path_hex)
                facing_dir = HexPosition.directions()[facing]
                
                # What path hexes does this candidate's GIVE cone cover?
                elev = max(0, elevations[idx])
                eff_sight = 3 + elev * elevation_mult  # default sight for supply calc
                fov = set(field_of_view(HexPosition.origin(), facing_dir, int(eff_sight)))
                
                covered = set()
                for ph in path:
                    rel = grid.index_to_hexposition(ph, idx)
                    if rel in fov:
                        covered.add(ph)
                
                if not covered:
                    continue
                
                tier = int(food_tiers[idx])
                
                # Keep best coverage per candidate hex
                if idx not in candidates or len(covered) > len(candidates[idx][2]):
                    candidates[idx] = (facing, tier, covered)
    
    if not candidates:
        return {}
    
    # --- Phase 2: Greedy set cover ---
    uncovered = set(path)
    selected = []  # (idx, facing, tier)
    remaining = dict(candidates)
    
    while uncovered and remaining:
        # Pick candidate with best score: coverage × tier
        best_idx = max(remaining, key=lambda i: (
            len(remaining[i][2] & uncovered) * remaining[i][1]
        ))
        facing, tier, covered = remaining.pop(best_idx)
        
        newly_covered = covered & uncovered
        if not newly_covered:
            continue
        
        selected.append((best_idx, facing, tier))
        uncovered -= newly_covered
    
    if not selected:
        return {}
    
    # --- Phase 3: Hungarian matching ---
    n_feeders = len(feeders)
    n_positions = len(selected)
    
    INF = 1e9
    cost = np.full((n_feeders, n_positions), INF)
    
    for i, piece in enumerate(feeders):
        if piece.location is None or piece.location < 0:
            continue
        for j, (pos_idx, _, _) in enumerate(selected):
            p = piece.pathfind(pos_idx, grid, elevations, countries)
            if not p:
                continue
            c = sum(_move_cost(elevations, p[k], p[k+1]) for k in range(len(p)-1))
            cost[i, j] = c
    
    row_ind, col_ind = linear_sum_assignment(cost)
    
    # --- Phase 4: Build instructions ---
    assignments = {}
    for i, j in zip(row_ind, col_ind):
        if cost[i, j] >= INF:
            continue
        
        piece = feeders[i]
        pos_idx, facing, _ = selected[j]
        
        # Path to position
        p = piece.pathfind(pos_idx, grid, elevations, countries)
        if len(p) < 2:
            continue
        rules = InstructionList.path_to_rules(p, grid, start_facing=piece.facing)
        
        # Rotate to face path
        path_end_facing = _facing_toward(grid, p[-2], p[-1]) if len(p) >= 2 else piece.facing
        diff = (facing - path_end_facing) % 6
        if diff <= 3:
            rules.extend([Instruction.ROT_R.value] * diff)
        else:
            rules.extend([Instruction.ROT_L.value] * (6 - diff))
        
        # Patrol loop: harvest then give
        rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
        
        assignments[piece.id] = InstructionList(rules, cursor=0, patrol=True)
    
    return assignments


can you build a demo. for gameboards maybe just assume elevation determines food yield for our example

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
# --- Board setup ---
board = PieceBoard.hilly(rings=8, radius=18, seed=42)
grid  = board.grid
elevs = board.elevations

# Fake food tiers from elevation: lowlands = fertile, mountains = barren
food_tiers = np.zeros(len(elevs), dtype=int)
for i, e in enumerate(elevs):
    if e <= 0:      food_tiers[i] = 0   # water
    elif e < 80:    food_tiers[i] = 6   # lush valley
    elif e < 140:   food_tiers[i] = 5   # plains
    elif e < 200:   food_tiers[i] = 3   # foothills
    elif e < 280:   food_tiers[i] = 1   # highlands
    else:           food_tiers[i] = 0   # peaks

print(f"Food tier distribution: {dict(zip(*np.unique(food_tiers, return_counts=True)))}")

# --- Find a good land start and a distant target ---
mid = grid.middle

# Queen starts NW of center
start_idx = None
for hp in [HexPosition(-4, -2, 6), HexPosition(-3, -2, 5), HexPosition(-3, -1, 4)]:
    idx = grid.hexposition_to_index(hp, mid)
    if idx >= 0 and elevs[idx] > 0:
        start_idx = idx
        break

# Target is SE
target_idx = None
for hp in [HexPosition(4, 2, -6), HexPosition(3, 2, -5), HexPosition(3, 1, -4)]:
    idx = grid.hexposition_to_index(hp, mid)
    if idx >= 0 and elevs[idx] > 0:
        target_idx = idx
        break

print(f"March: hex {start_idx} (elev={elevs[start_idx]:.0f}) → hex {target_idx} (elev={elevs[target_idx]:.0f})")

# --- Place the queen ---
queen = board.add_piece(start_idx, PieceType.QUEEN, facing=4, country_id=1,
                        instructions=InstructionList([Instruction.FORWARD.value], patrol=True))
queen.food = 3.0

# Get the queen's march path
march_path = queen.pathfind(target_idx, grid, elevs)
queen.instructions = queen.pathfind_to_rules(target_idx, grid, elevs)
print(f"March path: {len(march_path)} hexes, {len(queen.instructions.rules)} instructions")

# --- Scatter 10 pawns as potential feeders ---
feeder_offsets = [
    (-5, 0), (-4, 3), (-2, -3), (-1, 3), (0, -4),
    (1, 4), (2, -2), (3, -3), (4, 1), (5, -1)
]
feeders = []
for dq, dr in feeder_offsets:
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, mid)
    if idx < 0 or elevs[idx] <= 0:
        continue
    p = board.add_piece(idx, PieceType.PAWN, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 5.0
    p.sight = 3
    feeders.append(p)

print(f"{len(feeders)} feeders available")

# --- Run supply line ---
orders = supply_line(march_path, feeders, grid, elevs, food_tiers,
                     min_tier=3, max_offset=2)

print(f"\nAssigned {len(orders)} feeders to supply positions:")
for p in feeders:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name}: {len(p.instructions.rules)} instrs (patrol={p.instructions.patrol})")

# --- Render: sight cones + movement plans + facing bars ---
assigned = [p for p in feeders if p.id in orders]
show(NotStr(board.render_with_sight(
    assigned + [queen],
    num_turns=12,
    facing_only=True
)))
```

This gives you:
- **Queen's march** as a dashed arrow path across the map
- **Pawns converging** on fertile tiles flanking the route
- **Sight cones** (dotted) showing each feeder's GIVE coverage
- **Facing bars** confirming everyone points inward toward the path
- Unassigned pawns (too far away or no good position) stay idle

The valleys (low elevation = tier 5–6) should attract the feeders, while the river and mountains create natural gaps in coverage.

In [ ]:
@patch
def pathfind_to_rules(self: Piece, target: int, grid: HexGrid,
                      elevations: np.ndarray,
                      countries: np.ndarray = None) -> InstructionList:
    """Find path to target and convert to movement instructions."""
    path = self.pathfind(target, grid, elevations, countries)
    if len(path) < 2:
        return InstructionList([])
    
    rules = InstructionList.path_to_rules(path, grid, start_facing=self.facing)
    return InstructionList(rules, cursor=0, patrol=False)


@patch
def pathfind_to_food(self: Piece, grid: HexGrid,
                     elevations: np.ndarray, food_tiers: np.ndarray,
                     min_tier: int = 3,
                     countries: np.ndarray = None) -> InstructionList:
    """Find nearest tile with food >= min_tier and path there + harvest."""
    start = self.location
    if start is None or start < 0:
        return InstructionList([])
    
    pq = [(0.0, start)]
    visited = set()
    parent = {start: None}
    
    while pq:
        cost, current = heapq.heappop(pq)
        
        if current in visited:
            continue
        visited.add(current)
        
        if current != start and int(food_tiers[current]) >= min_tier:
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = parent[node]
            path.reverse()
            
            rules = InstructionList.path_to_rules(path, grid, start_facing=self.facing)
            rules.append(Instruction.HARVEST.value)
            return InstructionList(rules, cursor=0, patrol=False)
        
        for neighbor in grid.neighborsOf(current):
            if neighbor in visited or neighbor in grid.invalidRegion:
                continue
            if elevations[neighbor] <= 0:
                continue
            if countries is not None and countries[neighbor] < 0:
                continue
            
            edge_cost = _move_cost(elevations, current, neighbor)
            if neighbor not in parent:
                parent[neighbor] = current
                heapq.heappush(pq, (cost + edge_cost, neighbor))
    
    return InstructionList([])


In [ ]:
# --- Board setup ---
board = PieceBoard.hilly(rings=8, radius=18, seed=42)
grid  = board.grid
elevs = board.elevations
mid   = grid.middle

# Fake food tiers from elevation
food_tiers = np.zeros(len(elevs), dtype=int)
for i, e in enumerate(elevs):
    if e <= 0:      food_tiers[i] = 0
    elif e < 80:    food_tiers[i] = 6
    elif e < 140:   food_tiers[i] = 5
    elif e < 200:   food_tiers[i] = 3
    elif e < 280:   food_tiers[i] = 1
    else:           food_tiers[i] = 0

# --- Find start and target on the NORTH side (above river) ---
def find_land(offsets):
    for hp in offsets:
        idx = grid.hexposition_to_index(hp, mid)
        if idx >= 0 and elevs[idx] > 0:
            return idx
    return None

start_idx  = find_land([HexPosition(-5, -3, 8), HexPosition(-4, -3, 7), HexPosition(-3, -2, 5)])
target_idx = find_land([HexPosition(5, -3, -2), HexPosition(4, -3, -1), HexPosition(3, -2, -1)])

print(f"Start: hex {start_idx} (elev={elevs[start_idx]:.0f})")
print(f"Target: hex {target_idx} (elev={elevs[target_idx]:.0f})")

# --- Queen ---
queen = board.add_piece(start_idx, PieceType.QUEEN, facing=4, country_id=1,
                        instructions=InstructionList([]))
queen.food = 3.0
march_path = queen.pathfind(target_idx, grid, elevs)
queen.instructions = queen.pathfind_to_rules(target_idx, grid, elevs)
print(f"March path: {len(march_path)} hexes, {len(queen.instructions.rules)} instructions")

# --- Scatter feeders on the NORTH side ---
feeder_offsets = [
    (-4, -1), (-3, -3), (-1, -2), (0, -3), (1, -1),
    (2, -3), (3, -1), (-2, -1), (4, -2), (-1, -4)
]
feeders = []
for dq, dr in feeder_offsets:
    hp = HexPosition(dq, dr, -dq - dr)
    idx = grid.hexposition_to_index(hp, mid)
    if idx < 0 or elevs[idx] <= 0:
        continue
    p = board.add_piece(idx, PieceType.PAWN, facing=random.randint(0, 5), country_id=1,
                        instructions=InstructionList([]))
    p.food = 5.0
    p.sight = 3
    feeders.append(p)

print(f"{len(feeders)} feeders available")

# --- Supply line ---
orders = supply_line(march_path, feeders, grid, elevs, food_tiers,
                     min_tier=3, max_offset=2)

print(f"\nAssigned {len(orders)} feeders:")
for p in feeders:
    if p.id in orders:
        p.instructions = orders[p.id]
        print(f"  {p.name}: {len(p.instructions.rules)} instrs")

# --- Render ---
assigned = [p for p in feeders if p.id in orders]
show(NotStr(board.render_with_sight(
    assigned + [queen],
    num_turns=12,
    facing_only=True
)))


working?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

It's working! The SVG rendered with:

- **Queen** (knight icon) marching NW→E across the top of the map with ghost steps showing her 13-hex path
- **3 feeders assigned** out of 10 — the greedy set cover only needed 3 to get path coverage
- **Sight cones** (dark dotted hexes) showing the `#353911` overlay from `darkPrimary`
- **Facing bars** (white lines at hex edges) pointing inward toward the path
- **7 unassigned pawns** sitting idle (no instructions, too far away or not needed)

A few observations:

1. **Only 3 assigned** — that's actually the set cover being efficient! It picked 3 positions whose cones collectively cover the whole march route. The other 7 feeders just aren't needed.
2. **Queen starts at elev=1** — that's barely above water, so her path probably takes a circuitous route uphill. You might want a `min_elevation` filter on start candidates.
3. **The sight dots are visible** — `darkPrimary` at `0.35` opacity is reading well against the terrain.

Want to try it with more feeders needed (longer path, or `max_offset=1` to force tighter positioning)?